Cohere Labs

# Cohere Labs — ARC-AGI-2 Model Swarm — 4B parity floor + 2B drain + TRM

*Community notebook in the Cohere Labs Open Science spirit — not an official Cohere publication.*

| | |
|---|---|
| **Parity engine** | `sorokin/qwen3_4b_grids15_sft139` — NVARC TTT + DFS, byte-identical scoring cells, FULL 12 h window |
| **Drain engine** | `sorokin/qwen3_2b_grids15_sft141` — per-GPU drain chaser; generated solver (caps 600/270 s); own store `/kaggle/inference_outputs_2b` |
| **TRM engine** | `cpmpml/arc-prize-trm-031` (default ckpt `step_275886`) — TTT + 129-view voting on the first freed GPU, 64/32-aug dataset picked by window |
| **Merge** | post-selection guarded merge: attempt_1 overridden ONLY on 2B+TRM exact agreement (demote-never-delete); fill-only otherwise; ARCANA deterministic fallbacks LAST |
| **Machine** | Kaggle L4×4 (96 GB), offline, 12 h budget |
| **Crash ladder** | fallback placeholder @ cell 2 → snapshot every 600 s → endgame SIGTERM/SIGKILL @ GLOBAL_END−180 → layered final merge (validated atomic writes only) |
| **Determinism** | `PYTHONHASHSEED=0` + sha256 `stable_seed` rescore views — the approved fix for the 27.6–33.9 run-to-run band |
| **Time constants** | GLOBAL_END=t0+42600 · SWARM_END=t0+42120 · FINAL_MERGE_AT=t0+42180 · STARTER_KILL_AT=t0+42420 |
| **Parity deviations** | (a) `stable_seed` rescore aug (b) `resolve_model_path()` (c) `nprocs=max(1,device_count)` (d) `ARC_DEV_KEYS` dev gate (e) candidate-saturation early exit (f) 4-view rescore — **nothing else**, enforced by the build-time byte-diff audit |
| **Env knobs** | `ARC_SWARM_2B` `ARC_SWARM_TRM` (kill switches) · `ARC_MAIN_END_TIME` `ARC_DEV_KEYS` (dev) · `ARC2B_TASK_CAP` `ARC2B_DFS_CAP` · `TRM_CKPT_STEP` · `ARC_TRM_ENSEMBLE` `ARC_SELECTOR_AB` (experiments, OFF) · `VIZ_N` |

**Parity floor:** the 4B path never yields GPU time to the swarm — swarm jobs dispatch only after a
per-GPU drain event; no drain ⇒ zero swarm ⇒ the run degrades to pure baseline + determinism fix +
fallbacks. Extra families can only ADD candidates through the guarded merge.


## 1. Global time budget, environment & determinism

Baseline 10-minute buffer convention; `PYTHONHASHSEED=0`; swarm env knobs; stale-marker cleanup (G3); pynvml probe.

In [ ]:
# ---- 1. Constants & environment ----
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow

import os
import glob
import time
import shutil

t0 = time.time()

WALL            = t0 + 12 * 3600          # Kaggle hard kill; never scheduled against
GLOBAL_END      = t0 + 12 * 3600 - 600    # baseline convention, unchanged
global_end_time = GLOBAL_END              # baseline-compatible alias

# ARC_MAIN_END_TIME (dev lever only): relative seconds from t0, or an
# absolute epoch if >= 1e9.  Default = GLOBAL_END (full baseline window,
# the parity setting).
_main_env = os.getenv("ARC_MAIN_END_TIME")
if _main_env:
    _v = float(_main_env)
    MAIN_END = (t0 + _v) if _v < 1e9 else _v
elif os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    # v2: on the scored rerun the 240-task queue NEVER drains, so v1's
    # drain-triggered swarm never fired — the LB run was pure deterministic 4B.
    # Reserve a real tail window: v1's full-eval dev run measured the 2B family
    # alone uniquely rescuing 16/172 outputs once it got GPU time.
    # ARC_RERUN_SWARM_WINDOW=0 restores exact v1 rerun behavior.
    # v4 DEFAULT 0 = pure 4B for the whole 12h. Evidence: NVARC paper §4.4 —
    # TRM ensembled with their 4B scored 27.22 vs 27.22 without (zero gain);
    # our own v2(no swarm)=28.47 vs v3(swarm)=28.89 is inside the +/-2 LB noise
    # band. So the 90-min swarm tail costs ~17-30 4B task attempts and buys
    # nothing measurable. Set ARC_RERUN_SWARM_WINDOW=5400 to restore v3.
    _w = float(os.getenv("ARC_RERUN_SWARM_WINDOW", "0"))
    MAIN_END = GLOBAL_END - max(0.0, _w)
else:
    MAIN_END = GLOBAL_END

# STARTER_KILL_AT (G6): NOT a rare backstop -- with 240 tasks the queue never
# drains (workers claim puzzles until MAIN_END), so in EVERY competition rerun
# the starter is still alive here and the final cell's kill ladder fires,
# truncating the last ~3-8 min of decode across 4 GPUs (in-flight DFS batch +
# 8-view rescore that the baseline would finish inside its 600s margin).
# Expected cost ~0.1-0.4 tasks; the deliberate trade against the two
# historical 0.00 wall-timeout blowups (lesson 3).  Tunable: raising this to
# ~GLOBAL_END + 60..120 buys the window back at the price of wall margin
# (kill ladder 30s + layered final merge must still finish before WALL).
STARTER_KILL_AT = GLOBAL_END - 180
FINAL_MERGE_AT  = GLOBAL_END - 420        # orchestrator stops waiting
SWARM_END       = FINAL_MERGE_AT - 60     # --end-time for every 2B/TRM job

# Determinism (lesson 4): PYTHONHASHSEED=0 for the notebook process AND all
# children (companion of the approved stable_seed hunk (a) in arc_solver.py).
os.environ["PYTHONHASHSEED"] = "0"

# Swarm env knobs (spec §2) - defaults only; pre-set values win.
os.environ.setdefault("ARC_SWARM_2B", "0")       # v5: pure 4B (paper sec 4.4: TRM/2B union adds ~0 on hidden; v2 28.47 vs v3 28.89 = noise)
os.environ.setdefault("ARC_SWARM_TRM", "0")      # v5: pure 4B; forged checkpoint auto-detected (resolve_model_path prefers forge*)
os.environ.setdefault("ARC2B_TASK_CAP", "600")   # G2: halved caps for the ~2x 2B
os.environ.setdefault("ARC2B_DFS_CAP", "270")    # G2
os.environ.setdefault("ARC_TRM_ENSEMBLE", "0")   # experiment, OFF
os.environ.setdefault("ARC_SELECTOR_AB", "0")    # print-only AB, OFF

# Stale-marker cleanup (G3): unsloth import-serialization markers persist
# across interactive re-runs and would let all workers import concurrently.
for marker in glob.glob("/kaggle/worker*"):
    try:
        os.remove(marker)
        print(f"removed stale marker {marker}")
    except OSError:
        pass
shutil.rmtree("/kaggle/working/claims_2b", ignore_errors=True)
os.makedirs("/kaggle/working/trm_out", exist_ok=True)

# v2: dev saves default to the 4-puzzle smoke again (v1 already banked the
# full-eval validation). Set ARC_DEV_KEYS=all for a full 120-task dev run.

# ---- coverage knobs (v6). MEASURED on our own candidate pools: 6/6 solves were
# already present after the FIRST decode batch, and 35% of later batches add no
# new unique grid. So a task truncated to TTT + 1 batch keeps ~all of its solve
# probability, while an UNATTEMPTED task scores 0 with certainty. Therefore:
# spread the wall clock over all 240 tasks instead of letting slow tasks starve
# the tail, and stop paying for late batches.
os.environ.setdefault("ARC_ADAPTIVE_BUDGET", "1")     # fair-share per-puzzle cap
os.environ.setdefault("ARC_MAX_PUZZLE_SECONDS", "900")  # was a hard 1200
os.environ.setdefault("ARC_MIN_PUZZLE_SECONDS", "300")  # TTT (~206s) + 1 decode batch
os.environ.setdefault("ARC_NPROCS", "4")
os.environ.setdefault("ARC_ORDER", "short_first")     # bank cheap puzzles first
os.environ.setdefault("ARC_EARLY_EXIT", "1")          # stop a saturated test output
os.environ.setdefault("ARC_RESCORE_VIEWS", "4")       # 4 == 8 on selection (26/26)

# pynvml probe (drain confirmation; orchestration falls back to marker+grace)
try:
    import pynvml
    pynvml.nvmlInit()
    _n_gpu = pynvml.nvmlDeviceGetCount()
    pynvml.nvmlShutdown()
    print(f"pynvml OK: {_n_gpu} GPUs visible")
except Exception as _e:
    print(f"pynvml unavailable ({_e!r}) - drain confirmation uses marker + grace")

print(f"t0={t0:.0f}  GLOBAL_END=t0+{GLOBAL_END - t0:.0f}s  MAIN_END=t0+{MAIN_END - t0:.0f}s")
print(f"SWARM_END=t0+{SWARM_END - t0:.0f}s  FINAL_MERGE_AT=t0+{FINAL_MERGE_AT - t0:.0f}s  "
      f"STARTER_KILL_AT=t0+{STARTER_KILL_AT - t0:.0f}s")
print(f"knobs: 2B={os.environ['ARC_SWARM_2B']} TRM={os.environ['ARC_SWARM_TRM']} "
      f"2B caps={os.environ['ARC2B_TASK_CAP']}/{os.environ['ARC2B_DFS_CAP']}s "
      f"DEV_KEYS={os.getenv('ARC_DEV_KEYS', '<4-key smoke>')}")


## 2. Safety net & TRM prep

Fallback-enhanced placeholder `submission.json` BEFORE any GPU work (lesson 3); TRM checkpoint preflight; background 64/32-aug TRM dataset builds (G5).

In [ ]:
# ---- 2. Safety net & TRM prep (runs BEFORE any GPU work) ----
# Self-contained on purpose: no module written by a later cell is needed, so
# a valid submission.json exists on disk from minute one (lesson 3).  Slots
# get the deterministic ARCANA-port fallback grids instead of [[0]] -
# strictly better than the placeholder, replaced by every later snapshot.
import os
import sys
import json
import glob
import subprocess

import numpy as np

_rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
_root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
if not os.path.isdir(_root):
    _root = "/kaggle/input/arc-prize-2026-arc-agi-2"
_name = "arc-agi_test_challenges.json" if _rerun else "arc-agi_evaluation_challenges.json"
with open(os.path.join(_root, _name)) as _f:
    _challenges = json.load(_f)


def _fb_mode_color(grid):
    values, counts = np.unique(grid, return_counts=True)
    return int(values[int(np.argmax(counts))])


def _fb_crop(grid, bg=0):
    ys, xs = np.where(grid != bg)
    if len(ys) == 0:
        return grid.copy()
    return grid[ys.min():ys.max() + 1, xs.min():xs.max() + 1].copy()


def _fb_grids(test_input):
    ti = np.asarray(test_input, dtype=int)
    a1 = ti.tolist()
    for cand in (_fb_crop(ti, 0), _fb_crop(ti, _fb_mode_color(ti)),
                 np.full(ti.shape, _fb_mode_color(ti), dtype=int)):
        if (cand.ndim == 2 and 1 <= cand.shape[0] <= 30 and 1 <= cand.shape[1] <= 30
                and cand.tolist() != a1):
            return a1, cand.tolist()
    return a1, ([[0]] if a1 != [[0]] else [[1]])


def _grid_ok(grid):
    if not isinstance(grid, list) or not (1 <= len(grid) <= 30):
        return False
    width = None
    for row in grid:
        if not isinstance(row, list) or not (1 <= len(row) <= 30):
            return False
        if width is None:
            width = len(row)
        if len(row) != width or any((not isinstance(v, int)) or v < 0 or v > 9 for v in row):
            return False
    return True


_placeholder = {}
for _task_id, _task in _challenges.items():
    _rows = []
    for _pair in _task["test"]:
        try:
            _a1, _a2 = _fb_grids(_pair["input"])
        except Exception:
            _a1, _a2 = [[0]], [[0]]
        if not _grid_ok(_a1):
            _a1 = [[0]]
        if not _grid_ok(_a2):
            _a2 = [[0]]
        _rows.append({"attempt_1": _a1, "attempt_2": _a2})
    _placeholder[_task_id] = _rows

assert set(_placeholder) == set(_challenges)
with open("submission.json.tmp", "w") as _f:
    json.dump(_placeholder, _f)
with open("submission.json.tmp") as _f:
    json.load(_f)
os.replace("submission.json.tmp", "submission.json")
print(f"safety net: fallback-enhanced placeholder submission written "
      f"({len(_placeholder)} tasks)")

# ---- TRM preflight (never crash; TRM_CKPT_STEP override stays wired) ----
# FIRST-KAGGLE-RUN CHECKLIST -- the checkpoint layout of cpmpml/arc-prize-
# trm-031 is unverifiable locally, and this probe (like trm_driver.
# find_checkpoint) only accepts FILES whose basename matches step_<int>:
# flat step_N files and nested step_N/step_N both resolve; step_N/model.pt-
# style inner names return nothing and the TRM pass silently degrades to
# baseline+2B (trm_disabled marker, no crash).
#   1. Confirm this cell prints "TRM preflight OK: N checkpoints, default =
#      .../step_275886...".  If it prints "TRM DISABLED: no step_* checkpoint
#      file", read the tree dump it emits and set TRM_CKPT_STEP to the real
#      checkpoint path RELATIVE to /kaggle/input/arc-prize-trm-031 (wired
#      through trm_phase.py -> trm_driver.find_checkpoint via os.path.join).
#   2. Sanity-load one checkpoint once on Kaggle:
#      torch.load(path, map_location="cpu") -> keys start with "_orig_mod."
#      and include "_orig_mod.model.inner.puzzle_emb.weights".  The 2.16 GB
#      fp32 size is expected (puzzle-embedding-dominated), and
#      load_checkpoint's mean-reset of the size-mismatched puzzle_emb is the
#      designed path, not an error.
# Kaggle mounts inputs in either the flat layout (/kaggle/input/<slug>) or the
# namespaced layout (/kaggle/input/datasets/<owner>/<slug>) — v1 probed only the
# flat path and TRM self-disabled. Probe both, then glob as a last resort.
def _resolve_input(cands, marker):
    for _c in cands:
        if glob.glob(os.path.join(_c, marker)) or \
           glob.glob(os.path.join(_c, "**", marker), recursive=True):
            return _c
    _hits = glob.glob(os.path.join("/kaggle/input", "**", marker), recursive=True)
    return os.path.dirname(_hits[0]) if _hits else cands[0]

_TRM_CKPT_DIR = _resolve_input(
    ["/kaggle/input/arc-prize-trm-031",
     "/kaggle/input/datasets/cpmpml/arc-prize-trm-031"], "step_*")
_TRM_BUNDLE = _resolve_input(
    ["/kaggle/input/trm-bundle",
     "/kaggle/input/datasets/koushikrudra/trm-bundle"], "trm_driver.py")
os.environ["TRM_BUNDLE_DIR"] = _TRM_BUNDLE
os.environ["TRM_CKPT_DIR"] = _TRM_CKPT_DIR
print(f"TRM paths resolved: bundle={_TRM_BUNDLE}  ckpts={_TRM_CKPT_DIR}")
_trm_disabled_reason = None
_steps = []
for _p in glob.glob(os.path.join(_TRM_CKPT_DIR, "**", "step_*"), recursive=True):
    if os.path.isfile(_p):
        try:
            _steps.append((int(os.path.basename(_p).split("_")[1]), _p))
        except (IndexError, ValueError):
            continue
if not os.path.isdir(_TRM_BUNDLE):
    _trm_disabled_reason = f"bundle missing at {_TRM_BUNDLE}"
elif not _steps and not os.getenv("TRM_CKPT_STEP"):
    _trm_disabled_reason = f"no step_* checkpoint file under {_TRM_CKPT_DIR}"
    for _d in sorted(glob.glob(_TRM_CKPT_DIR + "/*"))[:20]:
        print("  tree:", _d)
        for _dd in sorted(glob.glob(_d + "/*"))[:5]:
            print("    ", _dd)

if _trm_disabled_reason:
    with open("/kaggle/working/trm_disabled", "w") as _f:
        _f.write(_trm_disabled_reason)
    print(f"TRM DISABLED: {_trm_disabled_reason}")
    if "no step_" in _trm_disabled_reason:
        print(f"  -> fix: read the tree dump above and set TRM_CKPT_STEP to the "
              f"checkpoint path relative to {_TRM_CKPT_DIR}, then re-run")
else:
    _steps.sort()
    if _steps:
        print(f"TRM preflight OK: {len(_steps)} checkpoints, default = {_steps[-1][1]}")
    else:
        _ovr = os.getenv("TRM_CKPT_STEP")
        _ovr_path = os.path.join(_TRM_CKPT_DIR, _ovr)
        _ovr_state = "resolves" if os.path.isfile(_ovr_path) else "DOES NOT RESOLVE"
        print(f"TRM preflight: relying on TRM_CKPT_STEP={_ovr} "
              f"({_ovr_state}: {_ovr_path})")

# ---- background TRM dataset builds: 64-aug AND 32-aug (G5) ----
# Deterministic (seed 42), single-core CPU, a few minutes each, fully
# overlapped with the 4B pass.  The dispatcher later picks by window size.
_BUILD_SCRIPT = '''\
import os, sys, time
try:
    import pydantic  # site-packages first (vendored pydantic_core is cp312-linux only)
except Exception:
    pass
BUNDLE = os.environ.get("TRM_BUNDLE_DIR", "/kaggle/input/trm-bundle")
sys.path.append(BUNDLE)
rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
if not os.path.isdir(root):
    root = "/kaggle/input/arc-prize-2026-arc-agi-2"
name = "arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json"
challenges = os.path.join(root, name)
from trm_build_data import build
for num_aug, out_dir in ((64, "/kaggle/working/trm_data_64"), (32, "/kaggle/working/trm_data_32")):
    t = time.time()
    try:
        build(challenges, out_dir, num_aug)
        with open(os.path.join(out_dir, ".done"), "w") as f:
            f.write("ok")
        print(f"[trm-data] {out_dir} built in {time.time()-t:.0f}s", flush=True)
    except Exception as e:
        print(f"[trm-data] BUILD FAILED for {out_dir}: {e!r}", flush=True)
'''
with open("/kaggle/working/trm_build_both.py", "w") as _f:
    _f.write(_BUILD_SCRIPT)

if not _trm_disabled_reason and os.getenv("ARC_SWARM_TRM", "1") != "0":
    _log = open("/kaggle/working/trm_build.log", "w")
    _env = dict(os.environ)
    _env.update({"PYTHONHASHSEED": "0", "OMP_NUM_THREADS": "2"})
    subprocess.Popen(
        ["nice", "-n", "10", sys.executable, "/kaggle/working/trm_build_both.py"],
        stdout=_log, stderr=subprocess.STDOUT, env=_env)
    print("TRM data build launched in background (64-aug then 32-aug; "
          "log: /kaggle/working/trm_build.log)")
else:
    print("TRM data build skipped (disabled)")


## 3. `arc_loader.py` — data, augmentation, submission builder

Byte-identical baseline module.

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

## 4. `arc_decoder.py` — candidate selection

Byte-identical baseline module.

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

## 5. `arc_solver.py` — per-puzzle test-time training + DFS decoding

Baseline + approved hunks (a) `stable_seed` rescore and (b) `resolve_model_path()` ONLY — keeps the baseline `turbo_dfs`/`calc_scores` perfpatch.

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import glob
import hashlib
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
NPROCS_ENV         = int(os.environ.get("ARC_NPROCS", "4"))
ADAPTIVE_BUDGET    = os.environ.get("ARC_ADAPTIVE_BUDGET", "1") not in ("0", "false", "False")
MAX_PUZZLE_SECONDS = int(os.environ.get("ARC_MAX_PUZZLE_SECONDS", "1200"))
MIN_PUZZLE_SECONDS = int(os.environ.get("ARC_MIN_PUZZLE_SECONDS", "240"))


def adaptive_cap(queue, end_time):
    """Fair share of remaining worker-seconds over remaining puzzles (hunk g)."""
    if not ADAPTIVE_BUDGET:
        return float(MAX_PUZZLE_SECONDS)
    try:
        pending = max(1, queue.qsize() - NPROCS_ENV + 1)
    except Exception:
        pending = 1
    fair = (end_time - time.time()) * NPROCS_ENV / pending
    return float(min(MAX_PUZZLE_SECONDS, max(MIN_PUZZLE_SECONDS, fair)))


USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


MODEL_PATH_CANDIDATES = [
    "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
    "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    "/kaggle/input/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
    "/kaggle/input/qwen3-4b-grids15-sft139/transformers/bfloat16/1",
    "/kaggle/input/qwen3-4b-grids15-sft139/Transformers/bfloat16/1",
]


def resolve_model_path():
    # ARC_MODEL_DIR: deploy a forged (continued-SFT) checkpoint instead of the
    # stock NVARC 4B. Explicit dir wins; otherwise any attached input whose dir
    # name starts with "forge" and carries a config.json is preferred, so a
    # re-forged model is picked up without editing this cell.
    env_dir = os.environ.get("ARC_MODEL_DIR")
    if env_dir and os.path.exists(os.path.join(env_dir, "config.json")):
        print(f"*** Using ARC_MODEL_DIR: {env_dir}")
        return env_dir
    for config_path in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        parent = os.path.basename(os.path.dirname(config_path)).lower()
        if parent.startswith("forge"):
            path = os.path.dirname(config_path)
            print(f"*** Using forged model path: {path}")
            return path
    for path in MODEL_PATH_CANDIDATES:
        if os.path.exists(os.path.join(path, "config.json")):
            print(f"*** Using model path: {path}")
            return path
    for config_path in glob.glob("/kaggle/input/**/config.json", recursive=True):
        normalized = config_path.lower()
        if "qwen3" in normalized and "grids15" in normalized:
            path = os.path.dirname(config_path)
            print(f"*** Using discovered model path: {path}")
            return path
    print("*** /kaggle/input roots:", sorted(glob.glob("/kaggle/input/*"))[:200])
    raise RuntimeError("Could not find qwen3_4b_grids15_sft139 config.json under /kaggle/input")


def stable_seed(text, modulo=1024**2):
    digest = hashlib.sha256(str(text).encode("utf-8")).hexdigest()
    return int(digest[:12], 16) % modulo


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=resolve_model_path(),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        puzzle_cap = adaptive_cap(queue, end_time)
        print(f"[Rank {rank}] {key} cap={puzzle_cap:.0f}s", flush=True)
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}
            # (e) candidate-saturation early exit -- see build_swarm_nb.py
            grids_by_bk = defaultdict(set)
            saturated_bk = set()
            early_exit = os.getenv("ARC_EARLY_EXIT", "1") != "0"

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > puzzle_cap or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                bk_batch = subkeys[0].split(".")[0]
                if early_exit and bk_batch in saturated_bk:
                    print(f"[Rank {rank}] skip saturated {bk_batch}")
                    continue
                n_grids_before = len(grids_by_bk[bk_batch])

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))
                        grids_by_bk[bk].add(grid_id[1])

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=stable_seed(bk))
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            n_rs = int(os.getenv("ARC_RESCORE_VIEWS", "4"))
                            augmented_scores = calc_scores(aug_queries[:min(4, n_rs)], aug_answers[:min(4, n_rs)], tokenizer, model)
                            if n_rs > 4:
                                augmented_scores = augmented_scores + calc_scores(aug_queries[4:n_rs], aug_answers[4:n_rs], tokenizer, model)
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

                if early_exit and len(grids_by_bk[bk_batch]) == n_grids_before:
                    saturated_bk.add(bk_batch)
                    print(f"[Rank {rank}] saturated {bk_batch} - skipping its remaining batches")

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

## 6. `starter.py` — L4×4 orchestration

Baseline + approved hunks (c) `nprocs=max(1,device_count)` and (d) `ARC_DEV_KEYS` dev-subset gate.

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    _keys = sorted(data.keys())
    if not rerun_mode:
        dev_keys = os.getenv("ARC_DEV_KEYS")
        if dev_keys is None:
            _keys = [k for k in _keys if k in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]]
        elif dev_keys != "all":
            _keys = [k for k in _keys if k in dev_keys.split(",")]

    def _cost(k):
        t = data[k]
        return (sum(len(r) * len(r[0]) for ex in t["train"] for r in (ex["input"], ex["output"]))
                + sum(len(ex["input"]) * len(ex["input"][0]) for ex in t["test"]))

    _order = os.environ.get("ARC_ORDER", "short_first")
    if _order == "short_first":
        _keys = sorted(_keys, key=lambda k: (_cost(k), k))
    elif _order == "long_first":
        _keys = sorted(_keys, key=lambda k: (-_cost(k), k))
    print(f"*** queueing {len(_keys)} puzzles, order={_order}", flush=True)
    for key in _keys:
        queue.put(key)
    nprocs = max(1, torch.cuda.device_count())
    print(f"CUDA device_count={torch.cuda.device_count()}, using nprocs={nprocs}")

    for _ in range(nprocs):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=nprocs)

## 7. `swarm_common.py` — grid utils, validators, fallbacks, ClaimQueue

New code only; diamond-reference submission validator + atomic writer; ARCANA deterministic fallbacks; lock-free cross-process ClaimQueue.

In [ ]:
%%writefile swarm_common.py
"""Shared pure-python/numpy utilities for the model-swarm notebook.

Everything in this module is NEW code (never imported by the byte-identical
baseline modules).  It provides:
  - grid canonicalisation / validation used by the guarded merge,
  - the diamond-reference submission validator + atomic writer (lesson 3),
  - the ARCANA-port deterministic fallback grids (strictly-positive EV,
    zero GPU cost, LAST priority in the merge),
  - ClaimQueue, a lock-free cross-process work queue that duck-types the
    mp.Manager().Queue() interface consumed by arc_solver.worker().
"""
import os
import json
import hashlib

import numpy as np


# ---------------------------------------------------------------------------
# grid helpers
# ---------------------------------------------------------------------------

def canon(grid):
    """Canonical hashable form of a grid (same idea as arc_decoder.hashable,
    but coerced to plain python ints so lists and ndarrays compare equal)."""
    return tuple(map(tuple, np.asarray(grid).astype(int).tolist()))


def valid_grid(grid):
    """2-D, dims 1..30, integral, values 0..9.

    NOTE: arc_loader.is_valid_solution does NOT check the value range; TRM's
    vocab (and any foreign family) makes the 0..9 check load-bearing.
    """
    try:
        a = np.asarray(grid)
    except Exception:
        return False
    if a.ndim != 2 or a.size == 0:
        return False
    if not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30):
        return False
    if not np.issubdtype(a.dtype, np.integer):
        if not np.issubdtype(a.dtype, np.floating):
            return False
        if not np.all(a == a.astype(int)):
            return False
        a = a.astype(int)
    return bool(np.all((a >= 0) & (a <= 9)))


def stable_seed(text, modulo=1024**2):
    """sha256-based seed; mirrors the approved hunk (a) in arc_solver.py."""
    digest = hashlib.sha256(str(text).encode("utf-8")).hexdigest()
    return int(digest[:12], 16) % modulo


# ---------------------------------------------------------------------------
# submission validation + atomic write (diamond final_cell.py L27-56 verbatim)
# ---------------------------------------------------------------------------

def _arc_grid_ok(grid):
    if not isinstance(grid, list) or not (1 <= len(grid) <= 30):
        return False
    width = None
    for row in grid:
        if not isinstance(row, list) or not (1 <= len(row) <= 30):
            return False
        if width is None:
            width = len(row)
        if len(row) != width:
            return False
        if any((not isinstance(value, int)) or value < 0 or value > 9 for value in row):
            return False
    return True


def _validate_submission(challenges, submission):
    assert set(submission) == set(challenges), "submission keys must match challenge keys"
    records = 0
    for key, challenge in challenges.items():
        rows = submission[key]
        assert isinstance(rows, list), f"{key}: value must be a list"
        assert len(rows) == len(challenge["test"]), f"{key}: wrong test record count"
        for row in rows:
            records += 1
            assert isinstance(row, dict), f"{key}: each record must be a dict"
            assert set(row) == {"attempt_1", "attempt_2"}, f"{key}: attempts must be attempt_1 and attempt_2"
            assert _arc_grid_ok(row["attempt_1"]), f"{key}: invalid attempt_1 grid"
            assert _arc_grid_ok(row["attempt_2"]), f"{key}: invalid attempt_2 grid"
    return records


def atomic_write_submission(submission, challenges, path="submission.json"):
    """Validate -> write .tmp -> reload -> re-validate -> os.replace.

    A failed validation NEVER replaces the previous good file (lesson 3).
    Returns True iff the on-disk submission.json was replaced.
    """
    try:
        records = _validate_submission(challenges, submission)
    except Exception as e:
        print(f"[swarm] submission REJECTED by validator, keeping previous file: {e!r}")
        return False
    tmp_path = path + ".tmp"
    try:
        with open(tmp_path, "w") as f:
            json.dump(submission, f)
        with open(tmp_path, "r") as f:
            reloaded = json.load(f)
        _validate_submission(challenges, reloaded)
    except Exception as e:
        print(f"[swarm] submission tmp write/reload FAILED, keeping previous file: {e!r}")
        try:
            os.remove(tmp_path)
        except OSError:
            pass
        return False
    os.replace(tmp_path, path)
    print(f"[swarm] submission.json updated ({len(submission)} tasks, {records} records)")
    return True


def all_base_keys(data):
    """Every `taskid_testidx` base key, enumerated from the ORIGINAL dataset
    (never from any model's outputs) so fill-only slots always exist."""
    return list(data.split_multi_replies().keys)


# ---------------------------------------------------------------------------
# deterministic fallbacks (ARCANA port; replaces only would-be [[0]] slots)
# ---------------------------------------------------------------------------

def get_mode_color(grid: np.ndarray) -> int:
    values, counts = np.unique(grid, return_counts=True)
    return int(values[int(np.argmax(counts))])


def crop_nonbackground(grid: np.ndarray, bg: int = 0) -> np.ndarray:
    ys, xs = np.where(grid != bg)
    if len(ys) == 0:
        return grid.copy()
    return grid[ys.min():ys.max() + 1, xs.min():xs.max() + 1].copy()


def deterministic_fallback_grids(test_input):
    """(attempt_1, attempt_2) heuristic grids for an otherwise-empty slot.

    attempt_1 = identity copy of the test input;
    attempt_2 = crop-to-nonbackground (bg=0) -> crop (bg=mode color) ->
                mode-color fill, first valid and != attempt_1.
    """
    ti = np.asarray(test_input, dtype=int)
    if valid_grid(ti):
        a1 = ti.copy()
    else:  # malformed input; never happens on real ARC data
        a1 = np.array([[0]], dtype=int)
    c1 = canon(a1)
    candidates = []
    try:
        candidates.append(crop_nonbackground(ti, bg=0))
    except Exception:
        pass
    try:
        candidates.append(crop_nonbackground(ti, bg=get_mode_color(ti)))
    except Exception:
        pass
    try:
        candidates.append(np.full(ti.shape, get_mode_color(ti), dtype=int))
    except Exception:
        pass
    a2 = None
    for cand in candidates:
        if valid_grid(cand) and canon(cand) != c1:
            a2 = np.asarray(cand, dtype=int)
            break
    if a2 is None:
        a2 = np.array([[0]], dtype=int) if c1 != ((0,),) else np.array([[1]], dtype=int)
    return a1, a2


# ---------------------------------------------------------------------------
# ClaimQueue -- lock-free work stealing for the 2B drain workers
# ---------------------------------------------------------------------------

class ClaimQueue:
    """Duck-types the mp queue consumed by arc_solver.worker (L339-347).

    get() atomically claims a task via os.open(O_CREAT | O_EXCL) on a
    per-task claim file, so any number of independently-launched worker
    processes can share one priority list with no Manager server.  Returns
    None when the list is exhausted (the worker's sentinel branch breaks).
    """

    def __init__(self, tasks, claims_dir):
        self.tasks = [str(t) for t in tasks]
        self.claims_dir = claims_dir
        os.makedirs(claims_dir, exist_ok=True)
        self._pos = 0

    def empty(self):
        return self._pos >= len(self.tasks)

    def get(self):
        while self._pos < len(self.tasks):
            key = self.tasks[self._pos]
            self._pos += 1
            claim_path = os.path.join(self.claims_dir, key)
            try:
                fd = os.open(claim_path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            except FileExistsError:
                continue
            except OSError:
                continue
            try:
                os.write(fd, str(os.getpid()).encode("utf-8"))
            finally:
                os.close(fd)
            return key
        return None


## 8. `swarm_merge.py` — tolerant loaders + guarded merge

The ONLY seam that may combine families (post-selection). Promotion requires 2B+TRM exact agreement; demote-never-delete; fill-only; fallbacks last.

In [ ]:
%%writefile swarm_merge.py
"""Guarded post-selection merge for the model swarm (the ONLY seam that may
combine model families -- decoder report §5.2).

Hard rules encoded here (empirical evidence, treat as law):
  - NEVER load swarm stores into the baseline decoder and NEVER pass
    run_name= on it: pooled loading re-ranks 4B answers via getter_kgmon
    vote counts (the 27.64 TRM-union-merge regression).
  - Baseline attempt_1 keeps its slot unless BOTH independent families
    (2B and TRM) exactly agree on a different grid; a demoted attempt_1 is
    never deleted (moves to attempt_2).
  - Fill-only otherwise; deterministic fallbacks (ARCANA port) run LAST and
    replace only would-be [[0]] placeholder slots.
  - With both family dicts empty and fallbacks disabled the merge output is
    attempt-for-attempt identical to the baseline ranking (the parity proof
    obligation, unit-tested in the diagnostics cell).
"""
import os
import bz2
import json
import pickle
from collections import OrderedDict

import numpy as np

from arc_decoder import ArcDecoder
import swarm_common
from swarm_common import canon, valid_grid


# ---------------------------------------------------------------------------
# tolerant loading (NEW code path -- the baseline decoder module stays intact)
# ---------------------------------------------------------------------------

class SortedTolerantDecoder(ArcDecoder):
    """ArcDecoder with a deterministic, partial-file-tolerant loader.

    Used by the snapshot loop (workers write bz2 pickles concurrently -- a
    half-written file must not kill the snapshot) and by every NEW loading
    path.  sorted(os.listdir(...)) removes the filesystem-enumeration
    tie-order nondeterminism (risk R4) in new code only; the baseline final
    load keeps baseline behavior.
    """

    def load_decoded_results(self, store, run_name=""):
        skipped = 0
        for key in sorted(os.listdir(store)):
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception:
                skipped += 1
                continue
            if not isinstance(outputs, list):
                skipped += 1
                continue
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample
        if skipped:
            print(f"[swarm] tolerant loader skipped {skipped} partial/bad files in {store}")


def load_family_ranked_2b(store_dir, data):
    """Family-internal kgmon ranking of the 2B drain store.

    The 2B solver inherits the exact {beam_score, score_aug, solution}
    pickle format, so a second decoder ranks it for free -- in its OWN
    decoder instance, never the baseline one.  Returns {} if the store is
    missing or empty.
    """
    if not store_dir or not os.path.isdir(store_dir) or not os.listdir(store_dir):
        return {}
    try:
        decoder = SortedTolerantDecoder(data.split_multi_replies(), n_guesses=2)
        decoder.load_decoded_results(store_dir)
        if not decoder.decoded_results:
            return {}
        return decoder.run_selection_algo()
    except Exception as e:
        print(f"[swarm] 2B family load failed harmlessly: {e!r}")
        return {}


def load_family_ranked_trm(pred_json, challenges):
    """Ranked TRM candidates as {base_key: [np.ndarray, ...]}.

    Hard checks (understand_trm §5):
      1. len(sub[task]) == len(challenges[task]["test"]) else DROP the task
         (the evaluator emits short/misaligned lists on zero-prediction
         pairs -- never index-trust);
      2. per-grid valid_grid() incl. the 0..9 value range;
      3. dedupe repeated attempts (the evaluator pads to K=10 by repeating
         attempt_1).
    """
    if not pred_json or not os.path.isfile(pred_json):
        return {}
    try:
        with open(pred_json, "r") as f:
            sub = json.load(f)
    except Exception as e:
        print(f"[swarm] TRM predictions unreadable: {e!r}")
        return {}
    ranked = {}
    dropped = 0
    for task_id, records in sub.items():
        challenge = challenges.get(task_id)
        if challenge is None:
            dropped += 1
            continue
        if not isinstance(records, list) or len(records) != len(challenge["test"]):
            dropped += 1
            continue
        for test_idx, record in enumerate(records):
            if not isinstance(record, dict):
                continue
            grids = []
            seen = set()
            for k in sorted(record.keys(), key=lambda s: (len(s), s)):  # attempt_1..attempt_10
                if not k.startswith("attempt_"):
                    continue
                g = record[k]
                if not valid_grid(g):
                    continue
                c = canon(g)
                if c in seen:
                    continue
                seen.add(c)
                grids.append(np.asarray(g, dtype=int))
            if grids:
                ranked[f"{task_id}_{test_idx}"] = grids
    if dropped:
        print(f"[swarm] TRM loader dropped {dropped} misaligned/unknown tasks")
    return ranked


# ---------------------------------------------------------------------------
# 2B drain priority (task level)
# ---------------------------------------------------------------------------

def build_2b_priority(decoded_results, data, store_dir):
    """P0 unanswered -> P1 single-candidate -> P2 weak (top votes <= 2).

    Re-attempting strong 4B answers is negative-EV merge pressure (C2), so
    nothing else enters the list.  Returns TASK ids (the queue unit).
    """
    p0, p1, p2 = [], [], []
    for task_id in sorted(data.keys):
        n_tests = len(data.queries[task_id]["test"])
        base_keys = [f"{task_id}_{i}" for i in range(n_tests)]
        unanswered = False
        single = False
        weak_votes = None
        for bk in base_keys:
            guesses = decoded_results.get(bk)
            if not guesses:
                unanswered = True
                continue
            groups = {}
            for sample in guesses.values():
                try:
                    groups.setdefault(canon(sample["solution"]), 0)
                    groups[canon(sample["solution"])] += 1
                except Exception:
                    continue
            if not groups:
                unanswered = True
                continue
            if len(groups) == 1:
                single = True
            top_votes = max(groups.values())
            if top_votes <= 2:
                weak_votes = top_votes if weak_votes is None else min(weak_votes, top_votes)
        if unanswered:
            p0.append(task_id)
        elif single:
            p1.append(task_id)
        elif weak_votes is not None:
            p2.append((weak_votes, task_id))
    p2_sorted = [t for _, t in sorted(p2)]
    priority = p0 + p1 + p2_sorted
    print(f"[swarm] 2B priority: P0={len(p0)} unanswered, P1={len(p1)} single, "
          f"P2={len(p2_sorted)} weak -> {len(priority)} tasks")
    return priority


# ---------------------------------------------------------------------------
# the guarded merge (pure function; spec §6)
# ---------------------------------------------------------------------------

def _first_valid(grids):
    for g in grids:
        if valid_grid(g):
            return np.asarray(g, dtype=int)
    return None


def _first_valid_excluding(grids, excluded):
    for g in grids:
        if valid_grid(g) and canon(g) not in excluded:
            return np.asarray(g, dtype=int)
    return None


def guarded_merge(baseline_ranked, families, all_bks, test_inputs, apply_fallbacks=True):
    """families: OrderedDict-like {"2B": ranked, "TRM": ranked} (either may
    be missing/empty).  Returns (results, provenance):
      results    {bk: [np.ndarray, ...]}  -- fill_submission-ready
      provenance {bk: {"attempt_1": src, "attempt_2": src, "promoted": bool,
                       "rule": str|None}}  src in {4B, 2B, TRM, AGREE, FB-*}
    """
    ranked_2b = families.get("2B") or {}
    ranked_trm = families.get("TRM") or {}
    results = {}
    provenance = {}

    for bk in all_bks:
        base = [np.asarray(g, dtype=int) for g in baseline_ranked.get(bk, []) if valid_grid(g)]
        a1 = base[0] if len(base) > 0 else None
        a2 = base[1] if len(base) > 1 else None
        src1 = "4B" if a1 is not None else None
        src2 = "4B" if a2 is not None else None
        promoted = False
        rule = None

        pool_2b = ranked_2b.get(bk, [])
        pool_trm = ranked_trm.get(bk, [])
        t2 = _first_valid(pool_2b)
        tt = _first_valid(pool_trm)

        # RULE 1 -- cross-family exact-agreement promotion (the ONLY rule that
        # can touch an occupied slot; requires BOTH families: the 2B shares
        # the 4B recipe, so a single family never overrides -- evidence C2).
        if t2 is not None and tt is not None and canon(t2) == canon(tt):
            G = canon(t2)
            if a1 is None:
                a1, src1 = t2, "AGREE"
                promoted, rule = True, "agree-fill"
            elif G == canon(a1):
                rule = "confirm"                      # confirmation; baseline wins ties
            elif a2 is not None and G == canon(a2):
                a1, a2 = a2, a1                       # swap; pair unchanged as a set
                src1, src2 = "AGREE", "4B"
                promoted, rule = True, "swap"
            else:
                a1, a2 = t2, a1                       # demote a1 -> a2, NEVER deleted
                src1, src2 = "AGREE", "4B"
                promoted, rule = True, "demote"

        # RULE 2 -- fill-only (never displaces an occupied slot)
        second_2b = _first_valid(pool_2b[1:]) if len(pool_2b) > 1 else None
        second_trm = _first_valid(pool_trm[1:]) if len(pool_trm) > 1 else None
        fill_order = [("2B", t2), ("TRM", tt), ("2B", second_2b), ("TRM", second_trm)]
        if a1 is None:
            for src, g in fill_order:
                if g is not None:
                    a1, src1 = g, src
                    break
        if a2 is None and a1 is not None:
            c1 = {canon(a1)}
            for src, g in fill_order:
                if g is not None and canon(g) not in c1:
                    a2, src2 = g, src
                    break

        # RULE 3 -- deterministic fallbacks (LAST; replaces only would-be
        # [[0]] placeholder slots, never a model candidate)
        if apply_fallbacks and (a1 is None or a2 is None):
            ti = test_inputs.get(bk)
            if ti is not None:
                fb1, fb2 = swarm_common.deterministic_fallback_grids(ti)
                if a1 is None:
                    a1, src1 = fb1, "FB-ID"
                    if a2 is None and canon(fb2) != canon(a1):
                        a2, src2 = fb2, "FB-CROP"
                elif a2 is None:
                    excluded = {canon(a1)}
                    fb = _first_valid_excluding([fb1, fb2], excluded)
                    if fb is not None:
                        a2, src2 = fb, "FB-ID" if canon(fb) == canon(fb1) else "FB-CROP"

        out = [g for g in (a1, a2) if g is not None]
        if out:
            results[bk] = out
        provenance[bk] = {
            "attempt_1": src1,
            "attempt_2": src2,
            "promoted": promoted,
            "rule": rule,
        }
    return results, provenance


# ---------------------------------------------------------------------------
# snapshot AND final-cell entry point
# ---------------------------------------------------------------------------

def build_final_submission(data, store_4b, store_2b, trm_json, tolerant=False,
                           include_2b=True, include_trm=True,
                           apply_fallbacks=True, write_provenance=True):
    """baseline selection -> family loads -> guarded merge -> get_submission.

    tolerant=False uses the untouched baseline ArcDecoder loader for the 4B
    store (baseline-exact final path); tolerant=True uses the sorted
    per-file-try/except loader (snapshot path, and the final cell's rescue
    rung after an endgame kill may leave partial files).
    """
    dec_cls = SortedTolerantDecoder if tolerant else ArcDecoder
    decoder = dec_cls(data.split_multi_replies(), n_guesses=2)
    if store_4b and os.path.isdir(store_4b):
        decoder.load_decoded_results(store_4b)
    baseline_ranked = decoder.run_selection_algo() if decoder.decoded_results else {}

    families = OrderedDict()
    if include_2b and os.getenv("ARC_SWARM_2B", "1") != "0":
        families["2B"] = load_family_ranked_2b(store_2b, data)
    if include_trm and os.getenv("ARC_SWARM_TRM", "1") != "0":
        families["TRM"] = load_family_ranked_trm(trm_json, data.queries)

    all_bks = swarm_common.all_base_keys(data)
    test_inputs = {}
    for task_id in data.keys:
        for i, pair in enumerate(data.queries[task_id]["test"]):
            test_inputs[f"{task_id}_{i}"] = np.asarray(pair["input"], dtype=int)

    merged, provenance = guarded_merge(
        baseline_ranked, families, all_bks, test_inputs, apply_fallbacks=apply_fallbacks)

    if write_provenance:
        try:
            prov_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
            with open(os.path.join(prov_dir, "merge_provenance.json"), "w") as f:
                json.dump(provenance, f)
        except Exception as e:
            print(f"[swarm] provenance write failed harmlessly: {e!r}")

    return data.get_submission(merged if merged else None)


## 9. `swarm_worker_2b.py` — 2B drain worker

One process per freed GPU; unsloth import re-serialized via `/kaggle/worker2b_{gpu}` markers (G4); ClaimQueue work-stealing.

In [ ]:
%%writefile swarm_worker_2b.py
"""2B drain worker: one process per freed GPU, launched by the orchestrator.

CUDA_VISIBLE_DEVICES is set in the environment BEFORE this process starts
(GPU isolation identical to the baseline starter).  The unsloth import race
is re-serialized with `/kaggle/worker2b_{gpu}` marker files (G4): this
process writes its marker only after `import arc_solver_2b` (which triggers
unsloth's global monkey-patching) completes; the orchestrator waits for the
marker before launching the next 2B worker.

Tasks come from the shared priority list via ClaimQueue -- lock-free
work-stealing, so later-drained GPUs join the same list and load-balance.
"""
import os
import json
import time
import argparse


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, required=True)
    parser.add_argument("--gpu", type=int, required=True)
    parser.add_argument("--priority-file", type=str,
                        default="/kaggle/working/priority_2b.json")
    parser.add_argument("--claims-dir", type=str,
                        default="/kaggle/working/claims_2b")
    args = parser.parse_args()

    print(f"[2B gpu{args.gpu}] start (CUDA_VISIBLE_DEVICES="
          f"{os.environ.get('CUDA_VISIBLE_DEVICES')!r}, "
          f"window={args.end_time - time.time():.0f}s)", flush=True)

    import torch
    torch.set_default_device("cpu")

    # Heavy import: unsloth patching + model classes.  Serialized by the
    # orchestrator via the marker below.
    import arc_solver_2b

    with open(f"/kaggle/worker2b_{args.gpu}", "w") as f:
        f.write("Ok")
    print(f"[2B gpu{args.gpu}] import done, marker written", flush=True)

    if time.time() > args.end_time:
        print(f"[2B gpu{args.gpu}] window closed during import, exiting", flush=True)
        return

    with open(args.priority_file, "r") as f:
        tasks = json.load(f)

    from swarm_common import ClaimQueue
    queue = ClaimQueue(tasks, args.claims_dir)

    arc_solver_2b.worker(args.gpu, queue, args.end_time)

    print(f"[2B gpu{args.gpu}] done!", flush=True)


if __name__ == "__main__":
    main()


## 10. Generate `arc_solver_2b.py`

Five asserted exactly-once substitutions on the emitted `arc_solver.py` — the 2B solver provably inherits the 4B recipe (incl. the perfpatch) with zero drift.

In [ ]:
# ---- 10. Generate arc_solver_2b.py (five asserted exactly-once substitutions) ----
# The 2B solver is GENERATED from the just-emitted arc_solver.py, so it
# provably inherits the whole 4B recipe (TTT config, DFS + the LB33.89
# perfpatch, stable_seed rescore, pickle format) with zero drift.  Each
# substitution must match exactly once or this cell raises (build gate).
import os
import hashlib

with open("arc_solver.py", "r", encoding="utf-8") as _f:
    _src = _f.read()

_CANDIDATES_4B = '''MODEL_PATH_CANDIDATES = [
    "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
    "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    "/kaggle/input/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
    "/kaggle/input/qwen3-4b-grids15-sft139/transformers/bfloat16/1",
    "/kaggle/input/qwen3-4b-grids15-sft139/Transformers/bfloat16/1",
]'''

_CANDIDATES_2B = '''MODEL_PATH_CANDIDATES = [
    "/kaggle/input/models/sorokin/qwen3_2b_grids15_sft141/transformers/bfloat16/1",
    "/kaggle/input/models/sorokin/qwen3_2b_grids15_sft141/Transformers/bfloat16/1",
    "/kaggle/input/qwen3_2b_grids15_sft141/transformers/bfloat16/1",
    "/kaggle/input/qwen3_2b_grids15_sft141/Transformers/bfloat16/1",
    "/kaggle/input/qwen3-2b-grids15-sft141/transformers/bfloat16/1",
    "/kaggle/input/qwen3-2b-grids15-sft141/Transformers/bfloat16/1",
]'''

_SUBS = [
    # (1) model path candidate list -> the 2B sibling
    (_CANDIDATES_4B, _CANDIDATES_2B),
    # (2) glob fallback filter (else the 4B model would match first) + the
    #     resolver's not-found message (same hunk-(b) region)
    ('if "qwen3" in normalized and "grids15" in normalized:',
     'if "2b" in normalized and "grids15" in normalized:'),
    ('raise RuntimeError("Could not find qwen3_4b_grids15_sft139 config.json under /kaggle/input")',
     'raise RuntimeError("Could not find qwen3_2b_grids15_sft141 config.json under /kaggle/input")'),
    # (3) own output dir (eval augs use fixed seed=2 => identical subkey
    #     filenames across models; sharing the dir would overwrite 4B pickles)
    ('dir_outputs = "/kaggle/inference_outputs"',
     'dir_outputs = "/kaggle/inference_outputs_2b"'),
    # (4) module-top cap knobs (G2: halved caps match the 2B's ~2x speed)
    ('logging.disable(logging.WARNING)',
     'logging.disable(logging.WARNING)\n\n'
     '_TASK_CAP = int(os.getenv("ARC2B_TASK_CAP", "600"))\n'
     '_DFS_CAP = int(os.getenv("ARC2B_DFS_CAP", "270"))'),
    # (4b) per-puzzle wall cap -> _TASK_CAP. The 4B solver now uses the
    # adaptive `puzzle_cap` (approved hunk g); the 2B drain keeps a fixed cap.
    ('if spend_time > puzzle_cap or time.time() > end_time:',
     'if spend_time > _TASK_CAP or time.time() > end_time:'),
    # (5) DFS wall cap 540 -> _DFS_CAP
    ('while time.time() - start_time < 540 and time.time() < end_time:',
     'while time.time() - start_time < _DFS_CAP and time.time() < end_time:'),
]

_out = _src
for _i, (_old, _new) in enumerate(_SUBS, 1):
    _n = _out.count(_old)
    assert _n == 1, (f"arc_solver_2b substitution #{_i} matched {_n} times "
                     f"(expected exactly 1): {_old[:70]!r}")
    _out = _out.replace(_old, _new)

with open("arc_solver_2b.py", "w", encoding="utf-8", newline="\n") as _f:
    _f.write(_out)
print("arc_solver_2b.py generated, sha256 =",
      hashlib.sha256(_out.encode("utf-8")).hexdigest()[:16])


## 11. `trm_phase.py` — TRM TTT + eval launcher

Site-pydantic-first import shim; sys.path APPEND of the bundle; env defaults; exit 0 ok / 2 ckpt-missing; never raises out.

In [ ]:
%%writefile trm_phase.py
"""TRM phase launcher: import-order shim + env defaults around trm_driver.

Import-order contract (understand_trm §7/§8 -- load-bearing):
  1. `import pydantic` from Kaggle site-packages FIRST (the bundle vendors a
     linux-cp312-only pydantic_core; the site version must win);
  2. sys.path.APPEND the bundle -- never insert -- so only the genuinely
     missing packages (adam_atan2_pytorch) resolve from the bundle.  The
     bundle's own sys.path.insert(0, ...) at trm_driver import time is
     defused because pydantic is already in sys.modules.

Exit codes: 0 ok / 2 checkpoint-or-bundle-missing (phase skipped) /
1 unexpected failure.  This script never raises out (lesson 3).

Env (set by the orchestrator dispatch): CUDA_VISIBLE_DEVICES, TRM_DATA_DIR,
TRM_CKPT_DIR, TRM_OUT_DIR, TRM_BATCH, TRM_BUDGET_S, TRM_EVAL_RESERVE_S,
TRM_END_TIME (absolute epoch deadline; when set it is recomputed into
TRM_BUDGET_S after the heavy imports, so the driver's relative
deadline = now + budget cannot extend past SWARM_END),
optional TRM_CKPT_STEP override, ARC_TRM_ENSEMBLE (experiment, OFF).
"""
import os
import sys
import time
import traceback

BUNDLE_DIR = os.getenv("TRM_BUNDLE_DIR", "/kaggle/input/trm-bundle")


def _prepare_paths():
    try:
        import pydantic  # noqa: F401  -- site-packages must win over the bundle
    except Exception as e:
        print(f"[trm-phase] site pydantic unavailable ({e!r}); bundle fallback will be tried")
    if not os.path.isdir(BUNDLE_DIR):
        print(f"[trm-phase] bundle missing at {BUNDLE_DIR} - phase skipped")
        return False
    if BUNDLE_DIR not in sys.path:
        sys.path.append(BUNDLE_DIR)
    return True


def _run_ensemble():
    """ARC_TRM_ENSEMBLE=1 (experiment, OFF by default): mirror of
    trm_driver.main() plus a pooled-vote eval of the second-latest
    checkpoint.  The ARC evaluator's aggregated_voting=True pools votes
    across evaluate() calls, so the last submission written is the
    cross-checkpoint ensemble.  The clock is checked before the second eval.
    """
    import glob
    import copy
    import shutil
    import torch
    import trm_driver
    from trm_eval_lib import (
        create_dataloader, init_train_state, train_batch, evaluate,
        create_evaluators, load_checkpoint,
    )
    from models.ema import EMAHelper

    data_dir = os.getenv("TRM_DATA_DIR", "/kaggle/working/trm_data")
    ckpt_dir = os.getenv("TRM_CKPT_DIR", "/kaggle/input/arc-prize-trm-031")
    out_dir = os.getenv("TRM_OUT_DIR", "/kaggle/working/trm_out")
    reserve_eval = float(os.getenv("TRM_EVAL_RESERVE_S", "900"))
    # Absolute deadline wins (computed here, i.e. AFTER the heavy imports
    # above): the schedule can never slip past SWARM_END by the import time.
    _end_env = os.getenv("TRM_END_TIME")
    if _end_env:
        deadline = float(_end_env)
    else:
        deadline = time.time() + float(os.getenv("TRM_BUDGET_S", "5400"))

    if not torch.cuda.is_available():
        os.environ.setdefault("DISABLE_COMPILE", "1")

    steps = []
    for path in glob.glob(os.path.join(ckpt_dir, "**", "step_*"), recursive=True):
        if os.path.isfile(path):
            try:
                steps.append((int(os.path.basename(path).split("_")[1]), path))
            except (IndexError, ValueError):
                continue
    steps.sort()
    if not steps:
        print(f"[trm-phase] no checkpoint under {ckpt_dir} - aborting phase")
        return 2
    ckpt_latest = steps[-1][1]
    ckpt_second = steps[-2][1] if len(steps) > 1 else None
    print(f"[trm-phase] ensemble: latest={ckpt_latest} second={ckpt_second}")

    config = trm_driver.build_config(data_dir, ckpt_latest, out_dir)
    torch.random.manual_seed(config.seed)

    epochs_per_iter = config.eval_interval
    train_loader, train_metadata = create_dataloader(
        config, "train", test_set_mode=False, epochs_per_iter=epochs_per_iter,
        global_batch_size=config.global_batch_size, rank=0, world_size=1)
    eval_loader, eval_metadata = create_dataloader(
        config, "test", test_set_mode=True, epochs_per_iter=1,
        global_batch_size=config.global_batch_size, rank=0, world_size=1)
    evaluators = create_evaluators(config, eval_metadata)

    train_state = init_train_state(config, train_metadata, rank=0, world_size=1)
    ema_helper = EMAHelper(mu=config.ema_rate)
    ema_helper.register(train_state.model)

    iter_id = 0
    while True:
        if time.time() > deadline - reserve_eval:
            print(f"[trm-phase] TTT budget reached after {iter_id} iters (step {train_state.step})")
            break
        train_state.model.train()
        stop = False
        for _set_name, batch, gbs in train_loader:
            train_batch(config, train_state, batch, gbs, rank=0, world_size=1)
            ema_helper.update(train_state.model)
            if time.time() > deadline - reserve_eval:
                stop = True
                break
        iter_id += 1
        if stop or train_state.step >= train_state.total_steps:
            break

    print("[trm-phase] FINAL EVAL (EMA, latest checkpoint)")
    train_state_eval = copy.deepcopy(train_state)
    train_state_eval.model = ema_helper.ema_copy(train_state_eval.model)
    train_state_eval.model.eval()
    evaluate(config, train_state_eval, eval_loader, eval_metadata, evaluators,
             rank=0, world_size=1, cpu_group=None)

    # Second checkpoint: pooled-vote eval only if the clock allows another
    # full eval pass (estimated at the same reserve).
    if ckpt_second is not None and time.time() < deadline - reserve_eval * 0.9:
        print("[trm-phase] SECOND EVAL (pooled votes, second-latest checkpoint)")
        try:
            config_second = trm_driver.build_config(data_dir, ckpt_second, out_dir)
            load_checkpoint(train_state_eval.model, config_second)
            train_state_eval.model.eval()
            evaluate(config, train_state_eval, eval_loader, eval_metadata, evaluators,
                     rank=0, world_size=1, cpu_group=None)
        except Exception as e:
            print(f"[trm-phase] second eval failed harmlessly: {e!r}")
    else:
        print("[trm-phase] skipping second eval (no second ckpt or clock)")

    subs = sorted(glob.glob(os.path.join(out_dir, "evaluator_ARC_step_*", "submission.json")))
    if subs:
        shutil.copy(subs[-1], os.path.join(out_dir, "trm_predictions.json"))
        print(f"[trm-phase] predictions -> {os.path.join(out_dir, 'trm_predictions.json')}")
        return 0
    print("[trm-phase] WARNING: evaluator produced no submission")
    return 1


def main():
    if not _prepare_paths():
        return 2
    os.environ.setdefault("TRM_OUT_DIR", "/kaggle/working/trm_out")
    os.environ.setdefault("TRM_CKPT_DIR", "/kaggle/input/arc-prize-trm-031")
    os.makedirs(os.environ["TRM_OUT_DIR"], exist_ok=True)
    try:
        if os.getenv("ARC_TRM_ENSEMBLE", "0") == "1":
            return _run_ensemble()
        import trm_driver
        # v3: OOM batch ladder. The v2 smoke run built TRM data, dispatched the
        # phase, and produced an EMPTY trm_out — the driver's batch-128 default
        # was tuned on bigger GPUs and is the prime OOM suspect on a 24GB L4.
        # Retry the whole driver at smaller TRM_BATCH before giving up; the
        # TRM_END_TIME -> TRM_BUDGET_S recompute happens per attempt so a
        # retry never thinks it still has the pre-crash budget.
        attempts = [os.environ.get("TRM_BATCH", "96"), "48", "24"]
        code = 1
        for _i, _b in enumerate(attempts):
            os.environ["TRM_BATCH"] = _b
            _end = os.getenv("TRM_END_TIME")
            if _end:
                _left = float(_end) - time.time()
                os.environ["TRM_BUDGET_S"] = str(max(60.0, _left))
                print(f"[trm-phase] attempt batch={_b}: TRM_END_TIME - now = "
                      f"{_left:.0f}s -> TRM_BUDGET_S={os.environ['TRM_BUDGET_S']}")
            try:
                code = trm_driver.main()
                code = int(code) if code is not None else 0
                break
            except Exception as _e:
                _oom = ("out of memory" in str(_e).lower()
                        or type(_e).__name__ == "OutOfMemoryError")
                print(f"[trm-phase] driver failed at TRM_BATCH={_b}: {_e!r} (oom={_oom})")
                if not _oom or _i == len(attempts) - 1:
                    traceback.print_exc()
                    code = 1
                    break
                import gc
                import torch
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        return code
    except SystemExit as e:
        try:
            return int(e.code or 0)
        except (TypeError, ValueError):
            return 1
    except Exception:
        print("[trm-phase] UNEXPECTED FAILURE:")
        traceback.print_exc()
        return 1


if __name__ == "__main__":
    sys.exit(main())


## 12. `orchestrator.py` — drain chaser, snapshots, kill ladders

Stdout tailing + pynvml drain confirmation; TRM-first dispatch policy; 600 s snapshot loop; SIGTERM→SIGKILL ladders. Meltdown degrades to baseline behavior.

In [ ]:
%%writefile orchestrator.py
"""Drain-chaser orchestrator (Design B, final spec §3/§7.2).

Contract with the parity floor:
  - The 4B main pass keeps the FULL baseline window (MAIN_END defaults to
    GLOBAL_END).  No code path starts any swarm job before a per-GPU 4B
    drain event; no drain => zero swarm GPU-seconds => the run degrades to
    pure baseline + determinism fix + fallbacks.
  - Swarm children are isolated Popen process groups with their own
    --end-time / TRM_BUDGET_S honoring SWARM_END, backstopped by a
    SIGTERM -> SIGKILL ladder.
  - The snapshot loop keeps a <=10-min-stale valid submission.json on disk
    at all times (exception-swallowed; a failed validation never replaces
    the previous good file).

Total orchestrator meltdown degrades to: placeholder + last snapshot + a
final cell that does not depend on this module (spec §7.4).
"""
import os
import re
import sys
import json
import time
import signal
import threading
import subprocess
import queue as pyqueue

import swarm_common
import swarm_merge
from arc_loader import ArcDataset

# ---- relative-duration constants (spec §1) --------------------------------
SNAPSHOT_S = 600            # snapshot-submission interval
TRM_MIN_S = 2400            # min window to dispatch TRM at all (else GPU -> 2B)
TRM_AUG64_MIN_S = 3900      # window >= this -> 64-aug data, reserve 1500
TRM_RESERVE_64 = 1500
TRM_RESERVE_32 = 900
TWO_B_MIN_S = 700           # min window for a 2B dispatch (model load + 1 task)
MARKER_WAIT_2B_S = 240      # G4: wait for the previous 2B import marker
PYNVML_FREE_BYTES = 20 * 1024**3   # drain confirmation threshold
NO_PYNVML_GRACE_S = 30      # marker + grace fallback when pynvml is absent
DEFAULT_NPROCS = 4

STORE_4B = "/kaggle/inference_outputs"
STORE_2B = "/kaggle/inference_outputs_2b"
TRM_OUT_DIR = "/kaggle/working/trm_out"
TRM_JSON = os.path.join(TRM_OUT_DIR, "trm_predictions.json")
TRM_CKPT_DIR = os.environ.get("TRM_CKPT_DIR", "/kaggle/input/arc-prize-trm-031")
TRM_DATA_64 = "/kaggle/working/trm_data_64"
TRM_DATA_32 = "/kaggle/working/trm_data_32"
TRM_DISABLED_MARKER = "/kaggle/working/trm_disabled"
PRIORITY_2B_JSON = "/kaggle/working/priority_2b.json"
CLAIMS_2B_DIR = "/kaggle/working/claims_2b"

# ---- module state (initialized by main / launch_starter) ------------------
_S = {
    "data": None,               # ArcDataset (original keys)
    "challenges": None,         # raw challenges dict for the validator
    "events": None,             # queue of rank-done events from the starter
    "swarm_procs": [],          # [(name, gpu, Popen)]
    "trm_dispatched": False,
    "last_2b_gpu": None,        # marker to wait on before the next 2B Popen
    "swarm_end": None,
    "priority_count": None,
}

_DONE_RE = re.compile(r"^\[Rank (\d+)\] done!\s*$")


def _comp_root():
    root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
    if not os.path.isdir(root):
        root = "/kaggle/input/arc-prize-2026-arc-agi-2"
    return root


def _load_data():
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    root = _comp_root()
    name = "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json"
    path = os.path.join(root, name)
    data = ArcDataset.from_file(path)
    with open(path, "r") as f:
        challenges = json.load(f)
    return data, challenges


def _reader_thread(proc, events, prefix):
    """Dedicated daemon reader: ALWAYS draining (prevents pipe-buffer
    stalls), echoes to the notebook log, pushes rank-done events."""
    def run():
        try:
            for line in proc.stdout:
                line = line.rstrip("\n")
                print(f"{prefix}{line}", flush=True)
                if events is not None:
                    m = _DONE_RE.match(line.strip())
                    if m:
                        events.put(int(m.group(1)))
        except Exception as e:
            print(f"{prefix}reader stopped: {e!r}", flush=True)
        finally:
            try:
                proc.stdout.close()
            except Exception:
                pass
    t = threading.Thread(target=run, daemon=True)
    t.start()
    return t


def _gpu_free_bytes(idx):
    try:
        import pynvml
        pynvml.nvmlInit()
        try:
            handle = pynvml.nvmlDeviceGetHandleByIndex(idx)
            info = pynvml.nvmlDeviceGetMemoryInfo(handle)
            return int(info.free)
        finally:
            pynvml.nvmlShutdown()
    except Exception:
        return None


def _gpu_count():
    try:
        import pynvml
        pynvml.nvmlInit()
        try:
            return int(pynvml.nvmlDeviceGetCount())
        finally:
            pynvml.nvmlShutdown()
    except Exception:
        return DEFAULT_NPROCS


# ---------------------------------------------------------------------------
# launches
# ---------------------------------------------------------------------------

def launch_starter(main_end):
    """Launch the byte-identical 4B starter as a killable process group.
    Env matches the baseline launch cell + PYTHONHASHSEED=0 (hunk (a)
    companion).  The stdout reader thread starts HERE, not in main(), so the
    pipe keeps draining even if the orchestrator loop crashes and the launch
    cell falls back to a plain wait (§7.4) -- a full pipe buffer would stall
    the 4B workers on print()."""
    env = dict(os.environ)
    env.update({
        "PYTHONHASHSEED": "0",
        "UNSLOTH_DISABLE_STATISTICS": "1",
        "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
        "OMP_NUM_THREADS": "12",
    })
    proc = subprocess.Popen(
        [sys.executable, "starter.py", "--end-time", str(float(main_end))],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors="replace", bufsize=1, start_new_session=True, env=env)
    _S["events"] = pyqueue.Queue()
    _reader_thread(proc, _S["events"], "")
    print(f"[orch] starter launched pid={proc.pid} end-time=t+{main_end - time.time():.0f}s",
          flush=True)
    return proc


def _swarm_env(gpu):
    env = dict(os.environ)
    env.update({
        "CUDA_VISIBLE_DEVICES": str(gpu),
        "PYTHONHASHSEED": "0",
        "OMP_NUM_THREADS": "4",      # bounds contention with still-live 4B workers
        "UNSLOTH_DISABLE_STATISTICS": "1",
        "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
    })
    return env


def dispatch_trm(gpu, budget_s):
    """Single-process TRM TTT+eval on GPU `gpu` (no torchrun/NCCL: batch 128
    fits one L4).  Picks the 64-aug dataset when the window allows, else the
    32-aug build (G5).  Returns the Popen or None if nothing is ready."""
    done64 = os.path.isfile(os.path.join(TRM_DATA_64, ".done"))
    done32 = os.path.isfile(os.path.join(TRM_DATA_32, ".done"))
    if budget_s >= TRM_AUG64_MIN_S and done64:
        data_dir, reserve = TRM_DATA_64, TRM_RESERVE_64
    elif done32:
        data_dir, reserve = TRM_DATA_32, TRM_RESERVE_32
    elif done64:
        data_dir, reserve = TRM_DATA_64, TRM_RESERVE_64
    else:
        print("[orch] TRM data not ready - GPU falls through to 2B", flush=True)
        return None
    env = _swarm_env(gpu)
    env.update({
        "TRM_DATA_DIR": data_dir,
        "TRM_CKPT_DIR": TRM_CKPT_DIR,
        "TRM_OUT_DIR": TRM_OUT_DIR,
        "TRM_BATCH": "96",
        "TRM_BUDGET_S": str(int(budget_s)),
        "TRM_EVAL_RESERVE_S": str(int(reserve)),
        # Absolute deadline (mirrors the 2B --end-time): trm_phase recomputes
        # TRM_BUDGET_S from this AFTER process spawn + torch/bundle imports
        # (~30-120s under load from still-live 4B workers), so trm_driver's
        # internal deadline lands exactly on SWARM_END instead of slipping
        # past it into the orchestrator's SIGTERM window.
        "TRM_END_TIME": str(float(_S["swarm_end"])),
    })
    # v3: persist TRM stdout to a file (the notebook log is not retrievable
    # via the API, which made the v2 TRM death undiagnosable post hoc).
    _trm_log = open("/kaggle/working/trm_phase.log", "a", buffering=1)
    proc = subprocess.Popen(
        ["nice", "-n", "10", sys.executable, "trm_phase.py"],
        stdout=_trm_log, stderr=subprocess.STDOUT,
        text=True, errors="replace", bufsize=1, start_new_session=True, env=env)
    print(f"[orch] TRM dispatched on gpu{gpu}: data={os.path.basename(data_dir)} "
          f"budget={budget_s:.0f}s reserve={reserve}s pid={proc.pid}", flush=True)
    return proc


def _ensure_priority_2b():
    """Build the 2B priority list once, from the latest tolerant snapshot of
    the 4B store.  Exception-safe: falls back to unanswered-only, then None."""
    if os.path.isfile(PRIORITY_2B_JSON):
        if _S["priority_count"] is None:
            try:
                with open(PRIORITY_2B_JSON) as f:
                    _S["priority_count"] = len(json.load(f))
            except Exception:
                _S["priority_count"] = 0
        return True
    data = _S["data"]
    try:
        decoder = swarm_merge.SortedTolerantDecoder(data.split_multi_replies(), n_guesses=2)
        if os.path.isdir(STORE_4B):
            decoder.load_decoded_results(STORE_4B)
        priority = swarm_merge.build_2b_priority(decoder.decoded_results, data, STORE_4B)
    except Exception as e:
        print(f"[orch] priority build failed ({e!r}) - falling back to unanswered-only", flush=True)
        try:
            covered = set()
            if os.path.isdir(STORE_4B):
                for fn in os.listdir(STORE_4B):
                    covered.add(fn.split(".")[0].split("_")[0])
            priority = [k for k in sorted(data.keys) if k not in covered]
        except Exception as e2:
            print(f"[orch] fallback priority build failed too: {e2!r}", flush=True)
            return False
    try:
        with open(PRIORITY_2B_JSON, "w") as f:
            json.dump(priority, f)
        _S["priority_count"] = len(priority)
    except Exception as e:
        print(f"[orch] priority write failed: {e!r}", flush=True)
        return False
    return True


def dispatch_2b(gpu, end_time):
    """2B drain worker on GPU `gpu`.  Serializes the unsloth import against
    the previous 2B dispatch via /kaggle/worker2b_{prev} (G4, timeout 240 s).
    Returns the Popen or None."""
    if not _ensure_priority_2b():
        return None
    if not _S["priority_count"]:
        print("[orch] 2B priority list is empty - nothing to dispatch", flush=True)
        return None
    try:
        claimed = len(os.listdir(CLAIMS_2B_DIR)) if os.path.isdir(CLAIMS_2B_DIR) else 0
    except OSError:
        claimed = 0
    if claimed >= _S["priority_count"]:
        print("[orch] all 2B priority tasks already claimed - skipping dispatch", flush=True)
        return None

    prev = _S["last_2b_gpu"]
    if prev is not None:
        marker = f"/kaggle/worker2b_{prev}"
        wait_until = time.time() + MARKER_WAIT_2B_S
        while not os.path.exists(marker) and time.time() < wait_until:
            if time.time() > end_time:
                return None
            time.sleep(2)
        if not os.path.exists(marker):
            print(f"[orch] previous 2B marker {marker} timed out - proceeding anyway", flush=True)

    proc = subprocess.Popen(
        ["nice", "-n", "10", sys.executable, "swarm_worker_2b.py",
         "--end-time", str(float(end_time)), "--gpu", str(gpu)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors="replace", bufsize=1, start_new_session=True,
        env=_swarm_env(gpu))
    _reader_thread(proc, None, f"[2B gpu{gpu}] ")
    _S["last_2b_gpu"] = gpu
    print(f"[orch] 2B dispatched on gpu{gpu} pid={proc.pid} "
          f"window={end_time - time.time():.0f}s", flush=True)
    return proc


def on_gpu_free(gpu, now):
    """Dispatch policy: first freed GPU -> TRM (if enabled, not disabled,
    window >= TRM_MIN_S, data ready) else 2B; subsequent GPUs -> 2B."""
    window = _S["swarm_end"] - now
    trm_enabled = (os.getenv("ARC_SWARM_TRM", "1") != "0"
                   and not os.path.exists(TRM_DISABLED_MARKER))
    if not _S["trm_dispatched"] and trm_enabled and window >= TRM_MIN_S:
        proc = dispatch_trm(gpu, int(window))
        if proc is not None:
            _S["trm_dispatched"] = True
            _S["swarm_procs"].append(("TRM", gpu, proc))
            return
    if os.getenv("ARC_SWARM_2B", "1") != "0" and window >= TWO_B_MIN_S:
        proc = dispatch_2b(gpu, _S["swarm_end"])
        if proc is not None:
            _S["swarm_procs"].append(("2B", gpu, proc))
            return
    print(f"[orch] gpu{gpu} freed but nothing to dispatch (window={window:.0f}s)", flush=True)


# ---------------------------------------------------------------------------
# snapshots & kills
# ---------------------------------------------------------------------------

def snapshot_submission():
    """Tolerant rebuild -> guarded merge -> atomic validated write.
    Exception-swallowed: a failing snapshot can never hurt the run.
    Duration is logged so the true 240-task merge cost (never seen in dev:
    smoke=4 tasks, ARC_DEV_KEYS=all=120) is visible in the rerun log."""
    try:
        t_start = time.time()
        submission = swarm_merge.build_final_submission(
            _S["data"], STORE_4B, STORE_2B, TRM_JSON, tolerant=True)
        swarm_common.atomic_write_submission(submission, _S["challenges"])
        print(f"[orch] snapshot merge+write took {time.time() - t_start:.1f}s", flush=True)
    except Exception as e:
        print(f"[orch] snapshot failed harmlessly: {e!r}", flush=True)


def kill_process_group(proc, term_wait_s=30):
    """SIGTERM the whole process group; SIGKILL after term_wait_s (G6)."""
    if proc is None or proc.poll() is not None:
        return
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    except Exception:
        try:
            proc.terminate()
        except Exception:
            pass
    deadline = time.time() + term_wait_s
    while time.time() < deadline:
        if proc.poll() is not None:
            return
        time.sleep(1)
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
    except Exception:
        try:
            proc.kill()
        except Exception:
            pass
    try:
        proc.wait(timeout=10)
    except Exception:
        pass


def kill_all_swarm(term_wait_s=30):
    for name, gpu, proc in _S["swarm_procs"]:
        if proc.poll() is None:
            print(f"[orch] killing {name} on gpu{gpu} (pid={proc.pid})", flush=True)
            kill_process_group(proc, term_wait_s=term_wait_s)


# ---------------------------------------------------------------------------
# main loop
# ---------------------------------------------------------------------------

def main(starter_proc, swarm_end, final_merge_at):
    """Tail the starter, chase per-GPU drains, snapshot every SNAPSHOT_S,
    stop waiting at final_merge_at (or earlier when everything is done)."""
    data, challenges = _load_data()
    _S.update({"data": data, "challenges": challenges, "swarm_end": swarm_end,
               "swarm_procs": [], "trm_dispatched": False,
               "last_2b_gpu": None, "priority_count": None})
    if _S["events"] is None:  # starter launched outside launch_starter()
        _S["events"] = pyqueue.Queue()
        _reader_thread(starter_proc, _S["events"], "")

    nprocs = _gpu_count()

    # Dev-only guard: in a non-rerun smoke run the swarm window is capped to
    # ARC_SWARM_DEV_WINDOW_S after the first drain so the default 4-key smoke
    # stays short.  Ignored entirely in the competition rerun.
    dev_window_cap = None
    if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        dev_window_cap = float(os.getenv("ARC_SWARM_DEV_WINDOW_S", "2700"))

    freed = set()               # ranks with a confirmed-free GPU (dispatched)
    pending = {}                # rank -> first-seen time (awaiting confirmation)
    dispatched = set()
    last_snapshot = time.time()
    starter_exit_snapshot_done = False

    while True:
        now = time.time()
        if now >= final_merge_at:
            print("[orch] FINAL_MERGE_AT reached - returning to the final cell", flush=True)
            break

        starter_done = starter_proc.poll() is not None

        # collect rank-done events from the starter stdout
        try:
            while True:
                rank = _S["events"].get_nowait()
                if rank not in freed and rank not in pending:
                    pending[rank] = now
                    print(f"[orch] rank {rank} reported done - awaiting GPU confirmation", flush=True)
        except pyqueue.Empty:
            pass

        # starter fully exited => every rank is free
        if starter_done:
            for rank in range(nprocs):
                if rank not in freed and rank not in pending:
                    pending[rank] = now - NO_PYNVML_GRACE_S  # no grace needed
            if not starter_exit_snapshot_done:
                print(f"[orch] starter exited (rc={starter_proc.returncode}) - extra snapshot",
                      flush=True)
                snapshot_submission()
                last_snapshot = time.time()
                starter_exit_snapshot_done = True

        # v3: reap dead swarm procs — a crashed TRM must not idle its GPU.
        _alive = []
        for _kind, _gpu, _proc in _S["swarm_procs"]:
            if _proc.poll() is None:
                _alive.append((_kind, _gpu, _proc))
                continue
            _rc = _proc.returncode
            if _kind == "TRM":
                _got = os.path.isfile(TRM_JSON)
                print(f"[orch] TRM gpu{_gpu} exited rc={_rc} preds={_got}", flush=True)
                if (not _got and os.getenv("ARC_SWARM_2B", "1") != "0"
                        and _S["swarm_end"] - now >= TWO_B_MIN_S):
                    _p2 = dispatch_2b(_gpu, _S["swarm_end"])
                    if _p2 is not None:
                        _alive.append(("2B", _gpu, _p2))
            elif _rc not in (0, None):
                print(f"[orch] {_kind} gpu{_gpu} exited rc={_rc}", flush=True)
        _S["swarm_procs"] = _alive

        # confirm pending frees (pynvml >= 20 GB free, else marker + grace)
        for rank in sorted(pending):
            first_seen = pending[rank]
            free_bytes = None if starter_done else _gpu_free_bytes(rank)
            confirmed = (
                starter_done
                or (free_bytes is not None and free_bytes >= PYNVML_FREE_BYTES)
                or (free_bytes is None and now - first_seen >= NO_PYNVML_GRACE_S)
            )
            if confirmed:
                del pending[rank]
                freed.add(rank)
                if rank not in dispatched:
                    dispatched.add(rank)
                    swarm_end_eff = _S["swarm_end"]
                    if dev_window_cap is not None:
                        swarm_end_eff = min(swarm_end_eff, now + dev_window_cap)
                        _S["swarm_end"] = swarm_end_eff
                    try:
                        on_gpu_free(rank, now)
                    except Exception as e:
                        print(f"[orch] on_gpu_free({rank}) failed harmlessly: {e!r}", flush=True)

        # reap swarm children
        for name, gpu, proc in _S["swarm_procs"]:
            rc = proc.poll()
            if rc is not None and not getattr(proc, "_reaped", False):
                proc._reaped = True
                print(f"[orch] {name} on gpu{gpu} exited rc={rc}", flush=True)

        # swarm end: kill ladder for stragglers
        if now >= _S["swarm_end"]:
            live = [(n, g, p) for n, g, p in _S["swarm_procs"] if p.poll() is None]
            if live:
                print(f"[orch] SWARM_END passed - killing {len(live)} swarm children", flush=True)
                kill_all_swarm()

        # periodic snapshot -- suppressed once the endgame starts (SWARM_END):
        # each snapshot is a FULL tolerant merge of every store (unbounded,
        # ~1-2+ min at 240 tasks) and the FINAL_MERGE_AT -> WALL window must
        # stay reserved for the kill ladder + the final cell's merge rungs.
        if now < _S["swarm_end"] and now - last_snapshot >= SNAPSHOT_S:
            snapshot_submission()
            last_snapshot = time.time()

        # exit early when there is nothing left to wait for
        swarm_alive = any(p.poll() is None for _, _, p in _S["swarm_procs"])
        if starter_done and not pending and not swarm_alive:
            all_dispatch_decided = dispatched >= set(range(nprocs))
            if all_dispatch_decided:
                print("[orch] all work finished - returning early", flush=True)
                break

        time.sleep(5)

    # Trailing snapshot ONLY on an early return (all work finished before
    # FINAL_MERGE_AT).  After FINAL_MERGE_AT the closing window belongs to
    # the final cell, which re-merges from the same stores anyway -- an
    # extra unbounded full merge here could stack with the kill ladder and
    # the final rungs and breach the 12 h wall (lesson 3).
    if time.time() < final_merge_at:
        snapshot_submission()


## 13. Launch — 4B main pass + swarm

The starter keeps the FULL baseline window; an orchestrator crash degrades to a plain wait (§7.4).

In [ ]:
# ---- 13. LAUNCH: 4B main pass (full parity window) + drain-chaser swarm ----
# The starter is the byte-identical baseline pipeline (hunks (c)/(d) only).
# The orchestrator tails its stdout, chases per-GPU drain events into 2B/TRM
# dispatches, and snapshots submission.json every 600 s.  An orchestrator
# crash degrades to a plain wait - it can never abort the notebook (§7.4).
import time
import traceback

import orchestrator

starter_proc = orchestrator.launch_starter(MAIN_END)
try:
    orchestrator.main(starter_proc, SWARM_END, FINAL_MERGE_AT)
except Exception as _e:
    print(f"[launch] orchestrator crashed ({_e!r}) - plain wait fallback")
    traceback.print_exc()
    try:
        while starter_proc.poll() is None and time.time() < FINAL_MERGE_AT:
            time.sleep(10)
    except Exception as _e2:
        print(f"[launch] fallback wait interrupted: {_e2!r}")

print(f"[launch] cell finished at t0+{time.time() - t0:.0f}s; "
      f"starter alive: {starter_proc.poll() is None}")


## 14. Final — endgame, guarded merge, `submission.json`

Unconditional. Endgame kill ladder (G6) → strict merge → tolerant merge → baseline-only → keep last snapshot; validated atomic writes only.

In [ ]:
# ---- 14. FINAL: endgame kill ladder + layered guarded merge + atomic write ----
# Unconditional (rerun AND dev).  Ladder (§7.3): endgame kill -> strict merge
# -> tolerant merge -> baseline-only + fallbacks -> leave last snapshot.
# A failed validation NEVER replaces the previous good submission.json.
import os
import json
import time
import traceback

import swarm_common
import swarm_merge
import orchestrator
from arc_loader import ArcDataset

_rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
_root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
if not os.path.isdir(_root):
    _root = "/kaggle/input/arc-prize-2026-arc-agi-2"
if _rerun:
    _ch_path = os.path.join(_root, "arc-agi_test_challenges.json")
    data = ArcDataset.from_file(_ch_path)
else:
    _ch_path = os.path.join(_root, "arc-agi_evaluation_challenges.json")
    data = ArcDataset.from_file(_ch_path)
    data = data.load_replies(os.path.join(_root, "arc-agi_evaluation_solutions.json"))
with open(_ch_path) as _f:
    challenges_raw = json.load(_f)

# --- 1. endgame: the starter must be dead before the final merge (G6).
# NOTE: this kill fires in EVERY competition rerun by design -- with 240
# tasks the queue never drains, so the starter is always still alive at
# STARTER_KILL_AT and gets SIGTERMed ~180s before its own --end-time (also
# losing the baseline's natural in-flight overrun past GLOBAL_END).  The
# cost (last minutes of decode on <=4 in-flight puzzles) is the deliberate
# trade against the two historical 0.00 wall-timeout blowups; see the
# STARTER_KILL_AT comment in cell 1 for the tunable.
_proc = globals().get("starter_proc")
if _proc is not None and _proc.poll() is None:
    print(f"[final] starter still alive - waiting until STARTER_KILL_AT "
          f"(t0+{STARTER_KILL_AT - t0:.0f}s)")
    while time.time() < STARTER_KILL_AT and _proc.poll() is None:
        time.sleep(5)
    if _proc.poll() is None:
        print("[final] endgame kill ladder: SIGTERM starter group, SIGKILL +30s")
        orchestrator.kill_process_group(_proc, term_wait_s=30)
try:
    orchestrator.kill_all_swarm(term_wait_s=30)
except Exception as _e:
    print(f"[final] swarm cleanup skipped: {_e!r}")

_STORE_4B = "/kaggle/inference_outputs"
_STORE_2B = "/kaggle/inference_outputs_2b"
_TRM_JSON = "/kaggle/working/trm_out/trm_predictions.json"

# --- 2. layered merge -> validate -> atomic replace
# WALL guard (lesson 3): each rung is a FULL unbounded merge (~1-4 min at
# 240 tasks; never measured in dev where smoke=4 / ARC_DEV_KEYS=all=120).
# When the 12 h hard wall is close, skip the strict rung and go straight to
# tolerant (<400s), and when it is imminent (<180s) run no rung at all --
# a valid snapshot submission.json is already on disk, and a kernel wall
# timeout scores 0.00 regardless of what is on disk.
_WALL_TS = t0 + 12 * 3600
_wrote = False
_left = _WALL_TS - time.time()
if _left < 400:
    print(f"[final] WALL guard: {_left:.0f}s to the wall - skipping the strict rung")
else:
    try:
        _t_rung = time.time()
        _submission = swarm_merge.build_final_submission(
            data, _STORE_4B, _STORE_2B, _TRM_JSON, tolerant=False)
        _wrote = swarm_common.atomic_write_submission(_submission, challenges_raw)
        print(f"[final] strict rung took {time.time() - _t_rung:.1f}s")
    except Exception as _e:
        print(f"[final] strict merge failed: {_e!r}")
        traceback.print_exc()
if not _wrote and _WALL_TS - time.time() >= 180:
    try:
        print("[final] rescue rung: tolerant merge (skips partial store files)")
        _t_rung = time.time()
        _submission = swarm_merge.build_final_submission(
            data, _STORE_4B, _STORE_2B, _TRM_JSON, tolerant=True)
        _wrote = swarm_common.atomic_write_submission(_submission, challenges_raw)
        print(f"[final] tolerant rung took {time.time() - _t_rung:.1f}s")
    except Exception as _e:
        print(f"[final] tolerant merge failed: {_e!r}")
if not _wrote and _WALL_TS - time.time() >= 180:
    try:
        print("[final] rescue rung: baseline-only + fallbacks")
        _submission = swarm_merge.build_final_submission(
            data, _STORE_4B, None, None, tolerant=True,
            include_2b=False, include_trm=False)
        _wrote = swarm_common.atomic_write_submission(_submission, challenges_raw)
    except Exception as _e:
        print(f"[final] baseline-only merge failed: {_e!r}")
if not _wrote:
    print("[final] leaving the last snapshot submission.json untouched "
          "(WALL guard or all rungs failed)")

# --- 3. reload + re-validate (the file that will actually be scored)
with open("submission.json") as _f:
    _reloaded = json.load(_f)
_records = swarm_common._validate_submission(challenges_raw, _reloaded)
print(f"[final] submission.json VALID: {len(_reloaded)} tasks, {_records} records")
if not _rerun:
    print("*** Reload score:", data.validate_submission(_reloaded))

# --- 4. pool export (Phase-A taxonomy raw material; dev AND rerun, non-fatal)
# v1 discarded /kaggle/inference_outputs, losing the candidate pools the
# near-miss taxonomy needs. Runs AFTER the validated submission is on disk.
try:
    import tarfile
    with tarfile.open("/kaggle/working/pools.tar.gz", "w:gz") as _tf:
        for _d in ("/kaggle/inference_outputs", "/kaggle/inference_outputs_2b",
                   "/kaggle/working/trm_out"):
            if os.path.isdir(_d):
                _tf.add(_d, arcname=os.path.basename(_d))
    print(f"[final] pool export: {os.path.getsize('/kaggle/working/pools.tar.gz')//1024} KB")
except Exception as _e:
    print(f"[final] pool export failed (non-fatal): {_e!r}")


# --- 5. coverage audit: the single number that explains this run's score
# An unattempted task is a certain 0. If coverage < 100% the score is
# BUDGET-limited, not model-limited, and must not be read as a model result.
try:
    import collections
    _touched = collections.Counter()
    for _d in ("/kaggle/inference_outputs", "/kaggle/inference_outputs_2b"):
        if os.path.isdir(_d):
            for _f in os.listdir(_d):
                _touched[_f.split(".")[0].rsplit("_", 1)[0]] += 1
    _n = len(challenges_raw)
    _seen = len(set(_touched) & set(challenges_raw))
    _outs = sum(len(t["test"]) for t in challenges_raw.values())
    print(f"[coverage] tasks={_n} test_outputs={_outs} "
          f"tasks_with_gpu_output={_seen} ({100*_seen/max(1,_n):.1f}%) "
          f"blank={_n - _seen}")
    if _n - _seen:
        print(f"[coverage] {_n - _seen} task(s) never reached the GPU -> each scores 0 "
              f"for certain. Lower ARC_MAX_PUZZLE_SECONDS or keep ARC_ADAPTIVE_BUDGET=1.")
except Exception as _e:
    print(f"[coverage] audit skipped: {_e!r}")


## 15. Diagnostics (dev only)

Merged-vs-baseline weighted score, per-family table, promotion win/loss (G7), guarded-merge parity unit test, pinned-hash audit (G8).

In [ ]:
# ---- 15. Diagnostics: swarm AB readout + parity audit (dev only) ----
try:
    import os
    import json
    import hashlib

    import numpy as np

    # G8: emitted-module hashes pinned at build time by build_swarm_nb.py.
    PINNED_MODULE_SHA256 = {
        "arc_decoder.py": "965cfd910777d9bbec9681c6c15c5e2ea3924569734248046f5e58f16cb28222",
        "arc_loader.py": "d01cd56167e534ae156706ef62ab11662fc7b403964614a7a55131940bde970c",
        "arc_solver.py": "ede77cea6ae38ced551aa81e268e0b2a2f885093d8e929222f7040fabd87cd9d",
        "arc_solver_2b.py": "32937214f1e508507f83a695dff5d6698b2fb871d2781a4c031138dd35a66191",
        "orchestrator.py": "863594bc3cf933fcc8bd6761af2ad87f0f9d49e452f023677d4ab16b59479707",
        "starter.py": "d7b80277e4a89f913b67d4b425d30063119a3d27f31b6a601a0dac18eabf22ac",
        "swarm_common.py": "b024bc43727bdc7fe216aa9c5f6596259751be8b3ce2cf8bf4c123d8f1d460d2",
        "swarm_merge.py": "6fdaea4636dc69dc4f2ecb764ebb0755719681f6620882d6983c52ccb25c1677",
        "swarm_worker_2b.py": "a94e4f6c81108220cfea6e276aeb6be1027ec51718e43d68fceb8140643b97d3",
        "trm_phase.py": "396b3b1c996564a3c5c35b97ae694e20857822a357c9f9a6eb405810b3be7e07",
    }
    for _fn, _expected in sorted(PINNED_MODULE_SHA256.items()):
        if not os.path.isfile(_fn):
            print(f"hash audit: {_fn} MISSING")
            continue
        with open(_fn, "rb") as _f:
            _actual = hashlib.sha256(_f.read()).hexdigest()
        _ok = "OK" if _actual == _expected else f"MISMATCH (got {_actual[:16]})"
        print(f"hash audit: {_fn:24s} {_expected[:16]}  {_ok}")

    # guarded_merge parity unit test (the §6 proof obligation) - synthetic
    # pools, no GPU, asserts printed not raised.
    import swarm_merge
    import swarm_common

    def _t(g):
        return np.asarray(g, dtype=int)

    _g1, _g2, _g3 = _t([[1, 2], [3, 4]]), _t([[5]]), _t([[6, 6]])
    _bks = ["t_0", "u_0", "v_0"]
    _tis = {bk: _t([[7, 0], [0, 7]]) for bk in _bks}
    _base = {"t_0": [_g1, _g2], "u_0": [_g1]}
    _res, _prov = swarm_merge.guarded_merge(_base, {}, _bks, _tis, apply_fallbacks=False)
    _p1 = (swarm_common.canon(_res["t_0"][0]) == swarm_common.canon(_g1)
           and swarm_common.canon(_res["t_0"][1]) == swarm_common.canon(_g2)
           and len(_res["u_0"]) == 1 and "v_0" not in _res)
    print(f"parity test (empty families, no fallbacks == baseline): "
          f"{'PASS' if _p1 else 'FAIL'}")
    _fam = {"2B": {"t_0": [_g3], "u_0": [_g3], "v_0": [_g3]},
            "TRM": {"t_0": [_g3], "v_0": [_g3]}}
    _res2, _prov2 = swarm_merge.guarded_merge(_base, _fam, _bks, _tis, apply_fallbacks=True)
    _p2 = (swarm_common.canon(_res2["t_0"][0]) == swarm_common.canon(_g3)   # demote
           and swarm_common.canon(_res2["t_0"][1]) == swarm_common.canon(_g1)
           and _prov2["t_0"]["rule"] == "demote"
           and swarm_common.canon(_res2["v_0"][0]) == swarm_common.canon(_g3)  # agree-fill
           and swarm_common.canon(_res2["u_0"][0]) == swarm_common.canon(_g1)  # 1 family: no override
           and _prov2["u_0"]["attempt_2"] == "2B")                             # fill-only
    print(f"promotion/fill test (agree demotes, never deletes): "
          f"{'PASS' if _p2 else 'FAIL'}")

    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        print("diagnostics: remaining sections skipped in competition rerun")
    else:
        from arc_loader import ArcDataset
        _root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
        if not os.path.isdir(_root):
            _root = "/kaggle/input/arc-prize-2026-arc-agi-2"
        _data = ArcDataset.from_file(f"{_root}/arc-agi_evaluation_challenges.json")
        _data = _data.load_replies(f"{_root}/arc-agi_evaluation_solutions.json")
        with open(f"{_root}/arc-agi_evaluation_solutions.json") as _f:
            _solutions = json.load(_f)
        with open("submission.json") as _f:
            _sub = json.load(_f)

        def _weighted(sub):
            tot, got = 0.0, 0.0
            for k, outs in _solutions.items():
                if k not in sub:
                    continue
                for i, gt in enumerate(outs):
                    tot += 1.0 / len(outs)
                    if i < len(sub[k]) and any(sub[k][i].get(f"attempt_{a}") == gt
                                               for a in (1, 2)):
                        got += 1.0 / len(outs)
            return got, tot

        _got_m, _tot = _weighted(_sub)
        _sub_base = swarm_merge.build_final_submission(
            _data, "/kaggle/inference_outputs", None, None, tolerant=True,
            include_2b=False, include_trm=False, apply_fallbacks=False,
            write_provenance=False)
        _got_b, _ = _weighted(_sub_base)
        print(f"weighted score  merged: {_got_m:.2f}/{int(round(_tot))}   "
              f"baseline-only: {_got_b:.2f}/{int(round(_tot))}   "
              f"delta: {_got_m - _got_b:+.2f}")

        # per-family coverage/correctness table
        _split = _data.split_multi_replies()
        _fams = {}
        _fams["2B"] = swarm_merge.load_family_ranked_2b("/kaggle/inference_outputs_2b", _data)
        _fams["TRM"] = swarm_merge.load_family_ranked_trm(
            "/kaggle/working/trm_out/trm_predictions.json", _data.queries)
        for _name, _ranked in _fams.items():
            _cov = len(_ranked)
            _top1 = _top2 = 0
            for _bk, _grids in _ranked.items():
                _gt = _split.replies.get(_bk)
                if not _gt:
                    continue
                _cgt = swarm_common.canon(_gt[0])
                if _grids and swarm_common.canon(_grids[0]) == _cgt:
                    _top1 += 1
                if any(swarm_common.canon(g) == _cgt for g in _grids[:2]):
                    _top2 += 1
            print(f"family {_name:4s}: {_cov:3d} base keys covered | "
                  f"top1 correct {_top1:3d} | top2 correct {_top2:3d}")

        # promotion AB readout (G7): fired / won / lost vs GT
        _prov_path = "/kaggle/working/merge_provenance.json"
        if os.path.isfile(_prov_path):
            with open(_prov_path) as _f:
                _provenance = json.load(_f)
            _fired = _won = _lost = _fills = _fbs = 0
            for _bk, _p in _provenance.items():
                _srcs = (_p.get("attempt_1"), _p.get("attempt_2"))
                _fills += sum(1 for s in _srcs if s in ("2B", "TRM"))
                _fbs += sum(1 for s in _srcs if s and s.startswith("FB"))
                if _p.get("promoted") and _p.get("rule") in ("swap", "demote"):
                    _fired += 1
                    _task, _idx = _bk.rsplit("_", 1)
                    _gt = (_solutions.get(_task) or [None] * 99)[int(_idx)]
                    if _gt is not None:
                        _k = _sub.get(_task)
                        if _k and int(_idx) < len(_k):
                            if _k[int(_idx)].get("attempt_1") == _gt:
                                _won += 1
                            elif _k[int(_idx)].get("attempt_2") == _gt:
                                pass  # survived in slot 2 - neutral
                            else:
                                _lost += 1
            print(f"promotions fired: {_fired} (a1 changed) | won: {_won} | "
                  f"lost: {_lost} | swarm fills: {_fills} | fallback slots: {_fbs}")
        else:
            print("no merge_provenance.json (merge never ran?)")

        # 2B drain queue composition
        if os.path.isfile("/kaggle/working/priority_2b.json"):
            with open("/kaggle/working/priority_2b.json") as _f:
                _prio = json.load(_f)
            _claims = (len(os.listdir("/kaggle/working/claims_2b"))
                       if os.path.isdir("/kaggle/working/claims_2b") else 0)
            print(f"2B queue: {len(_prio)} tasks prioritized, {_claims} claimed")
        else:
            print("2B queue: never built (no drain event)")
except Exception as e:
    print(f"diagnostics failed harmlessly: {e!r}")


## 16. Puzzle-wise gallery (dev only)

Provenance badges [4B]/[2B]/[TRM]/[AGREE]/[FB], per-family correctness dots, solved-first ordering, 6-state status matrix, VIZ_N cap.

In [ ]:
# ---- 16. Puzzle-wise gallery with per-family attribution (dev only) ----
try:
    import os
    import json
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        print("gallery: skipped in competition rerun")
    else:
        import numpy as np
        from matplotlib import colors
        import matplotlib.pyplot as plt

        import swarm_merge
        import swarm_common
        from arc_loader import ArcDataset

        _root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
        if not os.path.isdir(_root):
            _root = "/kaggle/input/arc-prize-2026-arc-agi-2"
        with open(f"{_root}/arc-agi_evaluation_challenges.json") as f:
            challenges = json.load(f)
        with open(f"{_root}/arc-agi_evaluation_solutions.json") as f:
            solutions = json.load(f)
        with open("submission.json") as f:
            sub = json.load(f)
        prov = {}
        if os.path.isfile("/kaggle/working/merge_provenance.json"):
            with open("/kaggle/working/merge_provenance.json") as f:
                prov = json.load(f)

        _data = ArcDataset.from_file(f"{_root}/arc-agi_evaluation_challenges.json")
        fam_ranked = {
            "4B": {},
            "2B": swarm_merge.load_family_ranked_2b("/kaggle/inference_outputs_2b", _data),
            "TRM": swarm_merge.load_family_ranked_trm(
                "/kaggle/working/trm_out/trm_predictions.json", challenges),
        }
        try:
            _dec = swarm_merge.SortedTolerantDecoder(_data.split_multi_replies(), n_guesses=2)
            if os.path.isdir("/kaggle/inference_outputs"):
                _dec.load_decoded_results("/kaggle/inference_outputs")
            if _dec.decoded_results:
                fam_ranked["4B"] = _dec.run_selection_algo()
        except Exception as e:
            print(f"gallery: 4B ranking unavailable ({e!r})")

        cmap = colors.ListedColormap(["#000000", "#0074D9", "#FF4136", "#2ECC40",
                                      "#FFDC00", "#AAAAAA", "#F012BE", "#FF851B",
                                      "#7FDBFF", "#870C25"])
        norm = colors.Normalize(vmin=0, vmax=9)

        def show(ax, arr, title):
            ax.imshow(np.asarray(arr, dtype=np.uint8), cmap=cmap, norm=norm)
            ax.set_title(title, fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])

        def fam_dot(name, bk, gt):
            grids = fam_ranked.get(name, {}).get(bk, [])
            if not grids:
                return f"{name}:-"
            if gt is not None and any(
                    swarm_common.canon(g) == swarm_common.canon(gt) for g in grids[:2]):
                return f"{name}:*"
            return f"{name}:o"

        def task_srcs(nm):
            out = set()
            for ti in range(len(challenges[nm]["test"])):
                p = prov.get(f"{nm}_{ti}", {})
                out.add(p.get("attempt_1")); out.add(p.get("attempt_2"))
            return {s for s in out if s}

        def solved_frac(nm):
            gts = solutions.get(nm)
            if not gts or nm not in sub:
                return 0.0
            hit = 0
            for ti, gt in enumerate(gts):
                if ti < len(sub[nm]) and any(sub[nm][ti].get(f"attempt_{a}") == gt
                                             for a in (1, 2)):
                    hit += 1
            return hit / max(1, len(gts))

        def attempted(nm):
            srcs = task_srcs(nm)
            return bool(srcs - {"FB-ID", "FB-CROP"})

        # solved-first ordering (G7), then attempted, then name
        ordered = sorted(challenges,
                         key=lambda nm: (-solved_frac(nm), not attempted(nm), nm))
        shown = 0
        for name in ordered:
            if name not in sub:
                continue
            if solved_frac(name) == 0.0 and not attempted(name):
                break
            task = challenges[name]
            for ti, tpair in enumerate(task["test"]):
                bk = f"{name}_{ti}"
                atts = sub[name][ti] if ti < len(sub[name]) else {}
                gt = solutions.get(name, [None] * len(task["test"]))[ti]
                p = prov.get(bk, {})
                ncols = len(task["train"]) * 2 + 3 + (1 if gt is not None else 0)
                fig, axes = plt.subplots(1, ncols, figsize=(1.5 * ncols, 1.9))
                k = 0
                for pair in task["train"]:
                    show(axes[k], pair["input"], "in"); k += 1
                    show(axes[k], pair["output"], "out"); k += 1
                show(axes[k], tpair["input"], "TEST in"); k += 1
                for ai in (1, 2):
                    g = atts.get(f"attempt_{ai}", [[0]])
                    mark = "" if gt is None else (" OK" if g == gt else " X")
                    src = p.get(f"attempt_{ai}") or "?"
                    show(axes[k], g, f"att{ai}[{src}]{mark}"); k += 1
                if gt is not None:
                    show(axes[k], gt, "GT")
                gtn = np.asarray(gt, dtype=int) if gt is not None else None
                dots = "  ".join(fam_dot(n, bk, gtn) for n in ("4B", "2B", "TRM"))
                fig.suptitle(f"{name}  [{dots}]   (* = top-2 correct, o = covered, - = none)",
                             fontsize=9)
                plt.tight_layout(); plt.show()
            shown += 1
            if shown >= int(os.getenv("VIZ_N", "6")):
                break

        # status matrix: 0 wrong / 1 solved-baseline / 2 solved-swarm-fill /
        # 3 solved-agreement / 4 no-GT / 5 fallback-only
        names = [n for n in challenges if n in sub]
        status = []
        for nm in names:
            srcs = task_srcs(nm)
            model_srcs = srcs - {"FB-ID", "FB-CROP"}
            gts = solutions.get(nm)
            if not gts:
                status.append(4); continue
            if not model_srcs:
                status.append(5); continue
            if solved_frac(nm) < 0.999:
                status.append(0); continue
            solving_srcs = set()
            for ti, gt in enumerate(gts):
                p = prov.get(f"{nm}_{ti}", {})
                for a in (1, 2):
                    if ti < len(sub[nm]) and sub[nm][ti].get(f"attempt_{a}") == gt:
                        solving_srcs.add(p.get(f"attempt_{a}"))
            if "AGREE" in solving_srcs:
                status.append(3)
            elif solving_srcs & {"2B", "TRM"}:
                status.append(2)
            else:
                status.append(1)
        ncol = 12
        nrow = (len(names) + ncol - 1) // ncol
        mat = np.full((nrow, ncol), -1, dtype=int)
        for idx, s in enumerate(status):
            mat[idx // ncol, idx % ncol] = s
        smap = colors.ListedColormap(["#FFFFFF",   # -1 pad
                                      "#E74C3C",   # 0 wrong
                                      "#2ECC40",   # 1 solved-baseline
                                      "#27AE60",   # 2 solved-swarm-fill
                                      "#F1C40F",   # 3 solved-agreement
                                      "#888888",   # 4 no GT
                                      "#E8E8E8"])  # 5 fallback-only
        fig, ax = plt.subplots(figsize=(ncol * 0.62, max(2.0, nrow * 0.62)))
        ax.imshow(mat + 1, cmap=smap, vmin=0, vmax=6)
        for idx, nm in enumerate(names):
            ax.text(idx % ncol, idx // ncol, nm[:4], ha="center", va="center", fontsize=5)
        ax.set_xticks([]); ax.set_yticks([])
        n_solved = len([s for s in status if s in (1, 2, 3)])
        n_att = len([s for s in status if s in (0, 1, 2, 3)])
        n_swarm = len([s for s in status if s in (2, 3)])
        ax.set_title(f"{n_solved}/{n_att} solved ({n_swarm} via swarm) | "
                     f"green=baseline dark-green=swarm-fill yellow=agreement "
                     f"red=wrong grey=no-GT light-grey=fallback-only", fontsize=8)
        plt.tight_layout(); plt.show()
except Exception as e:
    print(f"gallery failed harmlessly: {e!r}")


## 17. Experiments (OFF by default)

`ARC_TRM_ENSEMBLE`, `ARC_SELECTOR_AB` (print-only) — nothing touches the scoring path.

In [ ]:
# ---- 17. Experiments (env-gated, OFF by default) ----
# Nothing here may touch the scoring path.  Evidence discipline (C7): merge-
# rule or selector changes gate on a paired full-120 comparison from cell 15,
# never smoke-subset ABs; LB noise is +/-2 pts (1 TTT-sigma).
try:
    import os

    print("ARC_TRM_ENSEMBLE =", os.getenv("ARC_TRM_ENSEMBLE", "0"),
          " (1 => trm_phase pools votes over the two latest checkpoints; "
          "costs one extra eval pass inside the SAME TRM budget)")
    print("ARC_SELECTOR_AB  =", os.getenv("ARC_SELECTOR_AB", "0"),
          " (1 => print-only score_full_probmul_3 vs score_kgmon AB below; "
          "C6: 35.3 vs 34.8/116 on public eval - within noise, never a solo submission)")

    if (os.getenv("ARC_SELECTOR_AB", "0") == "1"
            and not os.getenv("KAGGLE_IS_COMPETITION_RERUN")):
        import swarm_merge
        from arc_loader import ArcDataset
        _root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
        if not os.path.isdir(_root):
            _root = "/kaggle/input/arc-prize-2026-arc-agi-2"
        _data = ArcDataset.from_file(f"{_root}/arc-agi_evaluation_challenges.json")
        _data = _data.load_replies(f"{_root}/arc-agi_evaluation_solutions.json")
        _dec = swarm_merge.SortedTolerantDecoder(_data.split_multi_replies(), n_guesses=2)
        if os.path.isdir("/kaggle/inference_outputs"):
            _dec.load_decoded_results("/kaggle/inference_outputs")
        if _dec.decoded_results:
            _dec.benchmark_selection_algos()   # prints BOTH selectors, print-only
        else:
            print("selector AB: no decoded results to benchmark")

    print("notes: TRM checkpoint override via TRM_CKPT_STEP; "
          "ARC_MAIN_END_TIME shortens the 4B window (dev drain forcing only); "
          "ARC_DEV_KEYS=all runs the full 120-task eval in dev mode.")
except Exception as e:
    print(f"experiments cell failed harmlessly: {e!r}")
